In [3]:
# ============================================================
# 전체 원본 데이터로 A/B 생산일 K-Means 검증
#
# 목적:
# AE C1/C2 1061개 사용 X
# 전체 원본 공정 데이터 사용
#
# 각 날짜의 공정 특성을 요약한 뒤
# K-Means(K=2)로 날짜 자체를 군집화
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# ============================================================
# 1. 파일 검색
# ============================================================

search_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
    Path.cwd().parent.parent.parent.parent
]

files = []

for root in search_roots:
    if root.exists():
        files += list(root.rglob("*.csv"))
        files += list(root.rglob("*.xlsx"))


# 분석 결과 파일 제외
exclude_words = [
    "summary",
    "candidate",
    "result",
    "output",
    "cycle",
    "error",
    "pattern",
    "kmeans",
    "ae_"
]

files = list(dict.fromkeys(files))

files = [
    f for f in files
    if not any(
        word in f.name.lower()
        for word in exclude_words
    )
]


# ============================================================
# 2. 컬럼 찾기 함수
# ============================================================

def normalize(x):
    return (
        str(x)
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace("(", "")
        .replace(")", "")
    )


def find_col(columns, keywords):

    for col in columns:

        c = normalize(col)

        for keyword in keywords:

            if normalize(keyword) in c:
                return col

    return None


# ============================================================
# 3. 원본 데이터 찾기
#
# 중요:
# 10,000행 이상인 데이터만 사용
# → AE 1061행 결과파일이 들어가는 것 방지
# ============================================================

data = None
source_file = None
source_sheet = None


for file in files:

    try:

        # ----------------------------------------------------
        # CSV
        # ----------------------------------------------------

        if file.suffix.lower() == ".csv":

            temp = pd.read_csv(file)

            force_col = find_col(
                temp.columns,
                ["weld force", "force"]
            )

            current_col = find_col(
                temp.columns,
                ["weld current", "current"]
            )

            voltage_col = find_col(
                temp.columns,
                ["weld voltage", "voltage"]
            )

            time_col = find_col(
                temp.columns,
                ["weld time", "weldtime"]
            )

            date_col = find_col(
                temp.columns,
                [
                    "working time",
                    "workingtime",
                    "date"
                ]
            )

            if (
                force_col is not None
                and current_col is not None
                and voltage_col is not None
                and time_col is not None
                and date_col is not None
                and len(temp) >= 10000
            ):

                data = temp.copy()
                source_file = file

                break


        # ----------------------------------------------------
        # Excel
        # ----------------------------------------------------

        elif file.suffix.lower() == ".xlsx":

            excel = pd.ExcelFile(file)

            for sheet in excel.sheet_names:

                try:

                    temp = pd.read_excel(
                        file,
                        sheet_name=sheet
                    )

                    force_col = find_col(
                        temp.columns,
                        ["weld force", "force"]
                    )

                    current_col = find_col(
                        temp.columns,
                        ["weld current", "current"]
                    )

                    voltage_col = find_col(
                        temp.columns,
                        ["weld voltage", "voltage"]
                    )

                    time_col = find_col(
                        temp.columns,
                        ["weld time", "weldtime"]
                    )

                    date_col = find_col(
                        temp.columns,
                        [
                            "working time",
                            "workingtime",
                            "date"
                        ]
                    )

                    if (
                        force_col is not None
                        and current_col is not None
                        and voltage_col is not None
                        and time_col is not None
                        and date_col is not None
                        and len(temp) >= 10000
                    ):

                        data = temp.copy()
                        source_file = file
                        source_sheet = sheet

                        break

                except:
                    pass

            if data is not None:
                break

    except:
        pass


# ============================================================
# 4. 못 찾았으면 중단
# ============================================================

if data is None:

    raise ValueError(
        """
전체 원본 데이터를 찾지 못했습니다.

조건:
- Force
- Current
- Voltage
- Weld Time
- 날짜
- 10,000행 이상

이 조건을 만족하는 원본 파일이 필요합니다.
"""
    )


# ============================================================
# 5. 실제 컬럼 다시 찾기
# ============================================================

force_col = find_col(
    data.columns,
    ["weld force", "force"]
)

current_col = find_col(
    data.columns,
    ["weld current", "current"]
)

voltage_col = find_col(
    data.columns,
    ["weld voltage", "voltage"]
)

time_col = find_col(
    data.columns,
    ["weld time", "weldtime"]
)

date_col = find_col(
    data.columns,
    [
        "working time",
        "workingtime",
        "date"
    ]
)


print("=" * 70)
print("사용 원본 데이터 확인")
print("=" * 70)

print("파일:", source_file)

if source_sheet is not None:
    print("시트:", source_sheet)

print("원본 행 개수:", len(data))

print("\n사용 컬럼")
print("Force     =", force_col)
print("Current   =", current_col)
print("Voltage   =", voltage_col)
print("Weld Time =", time_col)
print("Date      =", date_col)


# ============================================================
# 6. 분석용 데이터
# ============================================================

df = pd.DataFrame({

    "Force":
        pd.to_numeric(
            data[force_col],
            errors="coerce"
        ),

    "Current":
        pd.to_numeric(
            data[current_col],
            errors="coerce"
        ),

    "Voltage":
        pd.to_numeric(
            data[voltage_col],
            errors="coerce"
        ),

    "Weld_Time":
        pd.to_numeric(
            data[time_col],
            errors="coerce"
        ),

    "Date":
        pd.to_datetime(
            data[date_col],
            errors="coerce"
        ).dt.strftime("%m/%d")
})


df = df.dropna(
    subset=[
        "Force",
        "Current",
        "Voltage",
        "Weld_Time",
        "Date"
    ]
).reset_index(drop=True)


print("\n실제 분석 행:", len(df))

print(
    "날짜:",
    sorted(df["Date"].unique())
)


# ============================================================
# 7. 날짜별 특징 생성
#
# 날짜 이름 자체는 모델에 들어가지 않음.
# 날짜별 공정 특성만 사용.
# ============================================================

daily = (
    df.groupby("Date")
    .agg(

        Row_Count=("Force", "size"),

        Force_Mean=("Force", "mean"),
        Force_Std=("Force", "std"),

        Current_Mean=("Current", "mean"),
        Current_Std=("Current", "std"),

        Voltage_Mean=("Voltage", "mean"),
        Voltage_Std=("Voltage", "std"),

        WeldTime_Mean=("Weld_Time", "mean"),
        WeldTime_Std=("Weld_Time", "std")
    )
)


# ============================================================
# 8. Force 7~8 비율 추가
# ============================================================

df["Force_7_8"] = (
    (df["Force"] >= 7)
    &
    (df["Force"] < 9)
)


force78_rate = (
    df.groupby("Date")["Force_7_8"]
    .mean()
)


daily["Force_7_8_Rate"] = force78_rate


# ============================================================
# 9. Force 2.xxx 비율
# ============================================================

df["Force_2xxx"] = (
    (df["Force"] >= 2)
    &
    (df["Force"] < 3)
)


force2_rate = (
    df.groupby("Date")["Force_2xxx"]
    .mean()
)


daily["Force_2xxx_Rate"] = force2_rate


# ============================================================
# 10. K-Means 입력
#
# Row_Count는 생산량이므로 제외
# Date도 제외
# ============================================================

features = [

    "Force_Mean",
    "Force_Std",

    "Current_Mean",
    "Current_Std",

    "Voltage_Mean",
    "Voltage_Std",

    "WeldTime_Mean",
    "WeldTime_Std",

    "Force_7_8_Rate",
    "Force_2xxx_Rate"
]


X = daily[features].copy()


# 결측 발생 가능성 처리
X = X.fillna(0)


# ============================================================
# 11. 표준화
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# ============================================================
# 12. K-Means K=2
# ============================================================

model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=50
)


daily["Cluster"] = model.fit_predict(
    X_scaled
)


# ============================================================
# 13. Silhouette Score
# ============================================================

score = silhouette_score(
    X_scaled,
    daily["Cluster"]
)


# ============================================================
# 14. 기존 EDA A/B 가설
#
# 이것은 K-Means 학습에는 사용하지 않음.
# 학습 끝난 뒤 비교용.
# ============================================================

A_dates = [
    "03/24",
    "03/26",
    "03/30",
    "04/02",
    "04/07"
]


B_dates = [
    "03/25",
    "03/27",
    "03/31",
    "04/03"
]


daily["Expected_AB"] = np.where(

    daily.index.isin(A_dates),
    "A",

    np.where(
        daily.index.isin(B_dates),
        "B",
        "Unknown"
    )
)


# ============================================================
# 15. Force 7~8 존재 여부
# ============================================================

daily["Force_7_8_Exists"] = (
    daily["Force_7_8_Rate"] > 0
)


# ============================================================
# 16. 결과 출력
# ============================================================

print("\n" + "=" * 80)
print("전체 원본 데이터 K-Means A/B 검증")
print("=" * 80)

print(
    "Silhouette Score:",
    round(score, 4)
)


print("\n[Cluster별 날짜 수]")

print(
    daily["Cluster"]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 80)
print("날짜별 결과")
print("=" * 80)


for date, row in daily.iterrows():

    print(
        f"{date}"
        f" | 예상={row['Expected_AB']}"
        f" | KMeans={int(row['Cluster'])}"
        f" | Force7~8={row['Force_7_8_Exists']}"
        f" | Force7~8 비율={row['Force_7_8_Rate']*100:.2f}%"
        f" | Force평균={row['Force_Mean']:.3f}"
        f" | Current평균={row['Current_Mean']:.3f}"
        f" | Voltage평균={row['Voltage_Mean']:.3f}"
    )


# ============================================================
# 17. A/B × K-Means 결과
# ============================================================

known = daily[
    daily["Expected_AB"] != "Unknown"
]


cross = pd.crosstab(
    known["Expected_AB"],
    known["Cluster"]
)


print("\n" + "=" * 80)
print("EDA A/B 가설 × K-Means 결과")
print("=" * 80)

print(cross)


# ============================================================
# 18. Cluster별 특징
# ============================================================

cluster_character = (
    daily
    .groupby("Cluster")[
        [
            "Force_Mean",
            "Force_Std",
            "Current_Mean",
            "Current_Std",
            "Voltage_Mean",
            "Voltage_Std",
            "WeldTime_Mean",
            "WeldTime_Std",
            "Force_7_8_Rate",
            "Force_2xxx_Rate"
        ]
    ]
    .mean()
)


print("\n" + "=" * 80)
print("Cluster별 평균 특징")
print("=" * 80)


for c in cluster_character.index:

    r = cluster_character.loc[c]

    print(f"\nCluster {c}")

    print(
        "Force 평균:",
        round(r["Force_Mean"], 3)
    )

    print(
        "Force 표준편차:",
        round(r["Force_Std"], 3)
    )

    print(
        "Current 평균:",
        round(r["Current_Mean"], 3)
    )

    print(
        "Current 표준편차:",
        round(r["Current_Std"], 4)
    )

    print(
        "Voltage 평균:",
        round(r["Voltage_Mean"], 4)
    )

    print(
        "Voltage 표준편차:",
        round(r["Voltage_Std"], 4)
    )

    print(
        "Weld Time 평균:",
        round(r["WeldTime_Mean"], 3)
    )

    print(
        "Force 7~8 비율:",
        round(
            r["Force_7_8_Rate"] * 100,
            2
        ),
        "%"
    )

    print(
        "Force 2.xxx 비율:",
        round(
            r["Force_2xxx_Rate"] * 100,
            2
        ),
        "%"
    )


# ============================================================
# 19. 저장
# ============================================================

daily.to_csv(
    "kmeans_AB_daily_result.csv",
    encoding="utf-8-sig"
)


print("\n" + "=" * 80)
print("분석 완료")
print("=" * 80)

print("""
이번 결과에서 볼 것:

1. 원본 행 개수가 약 11,939개인지 확인

2. 날짜가
   03/24, 03/25, 03/26, 03/27,
   03/30, 03/31, 04/02, 04/03, 04/07
   모두 들어왔는지 확인

3. A 날짜 5개가 같은 Cluster에 들어가는지

4. B 날짜 4개가 다른 Cluster에 들어가는지

5. B쪽 Cluster에서 Force 7~8 비율이 실제로 높은지

→ 이게 맞으면
   EDA에서 발견한 A/B 두 생산 패턴이
   비지도 K-Means에서도 분리됐다고 볼 근거가 생김.
""")

사용 원본 데이터 확인
파일: c:\Kamp-2026\experiments\YSJ\YSJ-002\Welding Data Set_01.xlsx
시트: Raw data
원본 행 개수: 11939

사용 컬럼
Force     = weld force(bar)
Current   = weld current(kA)
Voltage   = weld Voltage(v)
Weld Time = weld time(ms)
Date      = working time

실제 분석 행: 11939
날짜: ['03/24', '03/25', '03/26', '03/27', '03/30', '03/31', '04/02', '04/03', '04/07']

전체 원본 데이터 K-Means A/B 검증
Silhouette Score: 0.602

[Cluster별 날짜 수]
Cluster
0    4
1    5
Name: count, dtype: int64

날짜별 결과
03/24 | 예상=A | KMeans=1 | Force7~8=False | Force7~8 비율=0.00% | Force평균=2.330 | Current평균=14.686 | Voltage평균=2.702
03/25 | 예상=B | KMeans=0 | Force7~8=True | Force7~8 비율=11.39% | Force평균=3.332 | Current평균=14.748 | Voltage평균=2.707
03/26 | 예상=A | KMeans=1 | Force7~8=False | Force7~8 비율=0.00% | Force평균=2.351 | Current평균=14.648 | Voltage평균=2.702
03/27 | 예상=B | KMeans=0 | Force7~8=True | Force7~8 비율=9.34% | Force평균=3.146 | Current평균=14.754 | Voltage평균=2.706
03/30 | 예상=A | KMeans=1 | Force7~8=False | Force7~8 비율=0.00% | Force평균

In [4]:
# ============================================================
# B 공정 내부 세부 공정패턴 탐색
#
# 목적:
# B공정(03/25, 03/27, 03/31, 04/03)만 추출
# → Force / Current / Voltage / Weld Time 사용
# → K=2~6 모두 비교
# → Silhouette Score로 군집 구조 확인
# → 선택된 K의 Cluster별 특성 확인
# → 4개 B 날짜에서 같은 Cluster가 반복되는지 확인
#
# 날짜는 K-Means 입력에 사용하지 않음
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# ============================================================
# 1. 원본 파일 찾기
# ============================================================

search_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
    Path.cwd().parent.parent.parent.parent
]

files = []

for root in search_roots:
    if root.exists():
        files += list(root.rglob("*.xlsx"))
        files += list(root.rglob("*.csv"))

files = list(dict.fromkeys(files))


# 분석 결과 파일 제외
exclude_words = [
    "summary",
    "candidate",
    "result",
    "output",
    "error",
    "pattern",
    "kmeans",
    "ae_"
]

files = [
    f for f in files
    if not any(
        word in f.name.lower()
        for word in exclude_words
    )
]


# ============================================================
# 2. 컬럼 찾기 함수
# ============================================================

def normalize(x):
    return (
        str(x)
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace("(", "")
        .replace(")", "")
    )


def find_col(columns, keywords):

    for col in columns:

        c = normalize(col)

        for keyword in keywords:

            if normalize(keyword) in c:
                return col

    return None


# ============================================================
# 3. 전체 원본 데이터 찾기
#
# 10,000행 이상 조건
# → AE 후보 1061행 같은 결과파일 제외
# ============================================================

data = None
source_file = None
source_sheet = None


for file in files:

    try:

        # ----------------------------------------------------
        # Excel
        # ----------------------------------------------------

        if file.suffix.lower() == ".xlsx":

            excel = pd.ExcelFile(file)

            for sheet in excel.sheet_names:

                try:

                    temp = pd.read_excel(
                        file,
                        sheet_name=sheet
                    )

                    force_col = find_col(
                        temp.columns,
                        ["weld force", "force"]
                    )

                    current_col = find_col(
                        temp.columns,
                        ["weld current", "current"]
                    )

                    voltage_col = find_col(
                        temp.columns,
                        ["weld voltage", "voltage"]
                    )

                    time_col = find_col(
                        temp.columns,
                        ["weld time", "weldtime"]
                    )

                    date_col = find_col(
                        temp.columns,
                        [
                            "working time",
                            "workingtime",
                            "date"
                        ]
                    )

                    if (
                        force_col is not None
                        and current_col is not None
                        and voltage_col is not None
                        and time_col is not None
                        and date_col is not None
                        and len(temp) >= 10000
                    ):

                        data = temp.copy()
                        source_file = file
                        source_sheet = sheet

                        break

                except:
                    pass

            if data is not None:
                break


        # ----------------------------------------------------
        # CSV
        # ----------------------------------------------------

        elif file.suffix.lower() == ".csv":

            temp = pd.read_csv(file)

            force_col = find_col(
                temp.columns,
                ["weld force", "force"]
            )

            current_col = find_col(
                temp.columns,
                ["weld current", "current"]
            )

            voltage_col = find_col(
                temp.columns,
                ["weld voltage", "voltage"]
            )

            time_col = find_col(
                temp.columns,
                ["weld time", "weldtime"]
            )

            date_col = find_col(
                temp.columns,
                [
                    "working time",
                    "workingtime",
                    "date"
                ]
            )

            if (
                force_col is not None
                and current_col is not None
                and voltage_col is not None
                and time_col is not None
                and date_col is not None
                and len(temp) >= 10000
            ):

                data = temp.copy()
                source_file = file

                break

    except:
        pass


if data is None:

    raise ValueError(
        "전체 원본 공정 데이터를 찾지 못했습니다."
    )


# ============================================================
# 4. 컬럼 다시 확인
# ============================================================

force_col = find_col(
    data.columns,
    ["weld force", "force"]
)

current_col = find_col(
    data.columns,
    ["weld current", "current"]
)

voltage_col = find_col(
    data.columns,
    ["weld voltage", "voltage"]
)

time_col = find_col(
    data.columns,
    ["weld time", "weldtime"]
)

date_col = find_col(
    data.columns,
    [
        "working time",
        "workingtime",
        "date"
    ]
)


print("=" * 70)
print("원본 데이터")
print("=" * 70)

print("파일:", source_file)

if source_sheet is not None:
    print("시트:", source_sheet)

print("원본 행:", len(data))


# ============================================================
# 5. 분석용 데이터 생성
# ============================================================

df = pd.DataFrame({

    "Force":
        pd.to_numeric(
            data[force_col],
            errors="coerce"
        ),

    "Current":
        pd.to_numeric(
            data[current_col],
            errors="coerce"
        ),

    "Voltage":
        pd.to_numeric(
            data[voltage_col],
            errors="coerce"
        ),

    "Weld_Time":
        pd.to_numeric(
            data[time_col],
            errors="coerce"
        ),

    "Date":
        pd.to_datetime(
            data[date_col],
            errors="coerce"
        ).dt.strftime("%m/%d")
})


df = df.dropna(
    subset=[
        "Force",
        "Current",
        "Voltage",
        "Weld_Time",
        "Date"
    ]
).reset_index(drop=True)


# ============================================================
# 6. B 공정 날짜만 추출
# ============================================================

B_dates = [
    "03/25",
    "03/27",
    "03/31",
    "04/03"
]


B = df[
    df["Date"].isin(B_dates)
].copy().reset_index(drop=True)


print("\n" + "=" * 70)
print("B 공정 데이터")
print("=" * 70)

print("B 전체 행:", len(B))


for date in B_dates:

    print(
        date,
        ":",
        len(
            B[B["Date"] == date]
        ),
        "행"
    )


# ============================================================
# 7. K-Means 입력 변수
#
# 날짜는 절대 사용하지 않음
# ============================================================

features = [
    "Force",
    "Current",
    "Voltage",
    "Weld_Time"
]


X = B[features].copy()


# ============================================================
# 8. 표준화
# ============================================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# ============================================================
# 9. K=2 ~ 6 전부 테스트
# ============================================================

scores = {}

models = {}


print("\n" + "=" * 70)
print("K별 Silhouette Score")
print("=" * 70)


for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = model.fit_predict(
        X_scaled
    )

    score = silhouette_score(
        X_scaled,
        labels
    )

    scores[k] = score
    models[k] = model

    print(
        f"K={k}: {score:.4f}"
    )


# ============================================================
# 10. 가장 높은 Silhouette K 선택
# ============================================================

best_k = max(
    scores,
    key=scores.get
)


best_score = scores[
    best_k
]


print("\n" + "=" * 70)
print("Silhouette 기준 결과")
print("=" * 70)

print(
    "가장 높은 K:",
    best_k
)

print(
    "Silhouette Score:",
    round(best_score, 4)
)


# ============================================================
# 11. 선택된 K로 최종 군집
# ============================================================

final_model = models[
    best_k
]


B["Cluster"] = final_model.labels_


# ============================================================
# 12. Cluster 개수
# ============================================================

cluster_count = (
    B["Cluster"]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 70)
print("Cluster별 데이터 개수")
print("=" * 70)


for cluster, count in cluster_count.items():

    rate = (
        count
        / len(B)
        * 100
    )

    print(
        f"Cluster {cluster}: "
        f"{count}개 "
        f"({rate:.2f}%)"
    )


# ============================================================
# 13. Cluster별 공정값 평균
# ============================================================

cluster_mean = (
    B.groupby("Cluster")[features]
    .mean()
)


cluster_std = (
    B.groupby("Cluster")[features]
    .std()
)


# Force 7~8
B["Force_7_8"] = (
    (B["Force"] >= 7)
    &
    (B["Force"] < 9)
)


# Force 2.xxx
B["Force_2xxx"] = (
    (B["Force"] >= 2)
    &
    (B["Force"] < 3)
)


force78 = (
    B.groupby("Cluster")["Force_7_8"]
    .mean()
    * 100
)


force2 = (
    B.groupby("Cluster")["Force_2xxx"]
    .mean()
    * 100
)


# ============================================================
# 14. Cluster별 특징 출력
# ============================================================

print("\n" + "=" * 70)
print("Cluster별 공정 특성")
print("=" * 70)


for cluster in sorted(
    B["Cluster"].unique()
):

    print(
        f"\n[Cluster {cluster}]"
    )

    print(
        "개수:",
        int(cluster_count.loc[cluster])
    )

    print(
        "Force 평균:",
        round(
            cluster_mean.loc[
                cluster,
                "Force"
            ],
            4
        )
    )

    print(
        "Force 표준편차:",
        round(
            cluster_std.loc[
                cluster,
                "Force"
            ],
            4
        )
    )

    print(
        "Current 평균:",
        round(
            cluster_mean.loc[
                cluster,
                "Current"
            ],
            4
        )
    )

    print(
        "Voltage 평균:",
        round(
            cluster_mean.loc[
                cluster,
                "Voltage"
            ],
            4
        )
    )

    print(
        "Weld Time 평균:",
        round(
            cluster_mean.loc[
                cluster,
                "Weld_Time"
            ],
            4
        )
    )

    print(
        "Force 2.xxx 비율:",
        round(
            force2.loc[cluster],
            2
        ),
        "%"
    )

    print(
        "Force 7~8 비율:",
        round(
            force78.loc[cluster],
            2
        ),
        "%"
    )


# ============================================================
# 15. 날짜 × Cluster 개수
# ============================================================

date_cluster_count = pd.crosstab(
    B["Date"],
    B["Cluster"]
)


# ============================================================
# 16. 날짜 × Cluster 비율
# ============================================================

date_cluster_rate = (
    pd.crosstab(
        B["Date"],
        B["Cluster"],
        normalize="index"
    )
    * 100
)


print("\n" + "=" * 80)
print("B 날짜별 Cluster 개수")
print("=" * 80)

print(
    date_cluster_count
)


print("\n" + "=" * 80)
print("B 날짜별 Cluster 비율 (%)")
print("=" * 80)

print(
    date_cluster_rate.round(2)
)


# ============================================================
# 17. 각 Cluster가 모든 B 날짜에서 존재하는지 확인
# ============================================================

print("\n" + "=" * 80)
print("Cluster 반복 여부")
print("=" * 80)


for cluster in sorted(
    B["Cluster"].unique()
):

    temp = B[
        B["Cluster"] == cluster
    ]

    dates_found = sorted(
        temp["Date"]
        .unique()
    )

    print(
        f"Cluster {cluster}: "
        f"{len(dates_found)}/4 날짜에서 존재"
    )

    print(
        "  날짜:",
        dates_found
    )


# ============================================================
# 18. 날짜별 Cluster 비율 유사성
#
# 각 B 날짜에서 비슷한 비율로
# Cluster가 반복되는지 확인
# ============================================================

print("\n" + "=" * 80)
print("날짜별 Cluster 구성")
print("=" * 80)


for date in B_dates:

    if date not in date_cluster_rate.index:
        continue

    print(f"\n{date}")

    for cluster in date_cluster_rate.columns:

        print(
            f"  Cluster {cluster}: "
            f"{date_cluster_rate.loc[date, cluster]:.2f}%"
        )


# ============================================================
# 19. 결과 저장
# ============================================================

B.to_csv(
    "B_process_kmeans_rows.csv",
    index=False,
    encoding="utf-8-sig"
)


date_cluster_count.to_csv(
    "B_process_cluster_count.csv",
    encoding="utf-8-sig"
)


date_cluster_rate.to_csv(
    "B_process_cluster_rate.csv",
    encoding="utf-8-sig"
)


# ============================================================
# 20. 최종 요약
# ============================================================

print("\n" + "=" * 80)
print("최종 요약")
print("=" * 80)

print(
    "B 공정 전체 행:",
    len(B)
)

print(
    "Silhouette 기준 선택 K:",
    best_k
)

print(
    "Silhouette Score:",
    round(best_score, 4)
)

print("""
해석 순서:

1. K=2~6 중 Silhouette가 어디서 가장 높은지 확인

2. 선택된 Cluster들이
   Force 2.xxx / Force 7~8 / Current / Voltage 등에서
   실제로 서로 다른 특징을 가지는지 확인

3. 각 Cluster가
   03/25, 03/27, 03/31, 04/03
   네 날짜에서 모두 반복되는지 확인

4. 네 날짜에서 Cluster 구성비까지 비슷하면
   → B공정 내부에서 반복되는 세부 공정패턴일 가능성

주의:
K-Means가 여러 Cluster를 만들었다는 사실만으로
실제 제조공정이 그 개수만큼 존재한다고 확정하지 않음.
공정값 특징 + 날짜 반복성을 같이 확인해야 함.
""")

원본 데이터
파일: c:\Kamp-2026\experiments\YSJ\YSJ-002\Welding Data Set_01.xlsx
시트: Raw data
원본 행: 11939

B 공정 데이터
B 전체 행: 5600
03/25 : 1352 행
03/27 : 1648 행
03/31 : 1800 행
04/03 : 800 행

K별 Silhouette Score
K=2: 0.4525
K=3: 0.4121
K=4: 0.4469
K=5: 0.4830
K=6: 0.4927

Silhouette 기준 결과
가장 높은 K: 6
Silhouette Score: 0.4927

Cluster별 데이터 개수
Cluster 0: 1295개 (23.12%)
Cluster 1: 240개 (4.29%)
Cluster 2: 2351개 (41.98%)
Cluster 3: 72개 (1.29%)
Cluster 4: 850개 (15.18%)
Cluster 5: 792개 (14.14%)

Cluster별 공정 특성

[Cluster 0]
개수: 1295
Force 평균: 2.3591
Force 표준편차: 0.2418
Current 평균: 14.7668
Voltage 평균: 2.704
Weld Time 평균: 70.9212
Force 2.xxx 비율: 97.53 %
Force 7~8 비율: 0.0 %

[Cluster 1]
개수: 240
Force 평균: 7.0295
Force 표준편차: 0.6227
Current 평균: 14.8605
Voltage 평균: 2.6582
Weld Time 평균: 71.6833
Force 2.xxx 비율: 0.0 %
Force 7~8 비율: 18.33 %

[Cluster 2]
개수: 2351
Force 평균: 2.3556
Force 표준편차: 0.2088
Current 평균: 14.7704
Voltage 평균: 2.7036
Weld Time 평균: 72.1229
Force 2.xxx 비율: 97.79 %
Force 7~8 비율: 0.0 %

[Cluster 3]
개수:

In [5]:
# ============================================================
# B공정 K-Means 추가 탐색
# K = 2 ~ 12
#
# 기존 셀에서 생성된
# X_scaled 사용
# ============================================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd


results = []

print("=" * 70)
print("B공정 K-Means K=2~12 추가 탐색")
print("=" * 70)


# ============================================================
# 1. K=2~12 전부 실행
# ============================================================

for k in range(2, 13):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    labels = model.fit_predict(X_scaled)

    silhouette = silhouette_score(
        X_scaled,
        labels
    )

    # 각 군집 크기
    counts = pd.Series(labels).value_counts()

    results.append({
        "K": k,
        "Silhouette": silhouette,
        "Smallest_Cluster": counts.min(),
        "Largest_Cluster": counts.max()
    })

    print(
        f"K={k:2d} | "
        f"Silhouette={silhouette:.4f} | "
        f"최소 군집={counts.min():4d} | "
        f"최대 군집={counts.max():4d}"
    )


# ============================================================
# 2. 결과표
# ============================================================

result_df = pd.DataFrame(results)


print("\n" + "=" * 70)
print("Silhouette 순위")
print("=" * 70)

print(
    result_df
    .sort_values(
        "Silhouette",
        ascending=False
    )
    .to_string(index=False)
)


# ============================================================
# 3. 최고 점수 K
# ============================================================

best_row = result_df.loc[
    result_df["Silhouette"].idxmax()
]

best_k = int(best_row["K"])


print("\n" + "=" * 70)
print("현재 최고 K")
print("=" * 70)

print("K =", best_k)
print(
    "Silhouette =",
    round(
        best_row["Silhouette"],
        4
    )
)


# ============================================================
# 4. 최고 K로 다시 학습
# ============================================================

final_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=50
)

B["Cluster_extended"] = final_model.fit_predict(
    X_scaled
)


# ============================================================
# 5. Cluster별 개수
# ============================================================

count = (
    B["Cluster_extended"]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 70)
print(f"K={best_k} Cluster별 개수")
print("=" * 70)


for c, n in count.items():

    print(
        f"Cluster {c}: "
        f"{n}개 "
        f"({n / len(B) * 100:.2f}%)"
    )


# ============================================================
# 6. Cluster별 공정 특성
# ============================================================

summary = (
    B.groupby("Cluster_extended")
    .agg(
        Count=("Force", "size"),

        Force_mean=("Force", "mean"),
        Force_std=("Force", "std"),

        Current_mean=("Current", "mean"),
        Current_std=("Current", "std"),

        Voltage_mean=("Voltage", "mean"),
        Voltage_std=("Voltage", "std"),

        WeldTime_mean=("Weld_Time", "mean"),
        WeldTime_std=("Weld_Time", "std")
    )
)


# Force 조건 추가
force2_rate = (
    B.assign(
        condition=(
            (B["Force"] >= 2) &
            (B["Force"] < 3)
        )
    )
    .groupby("Cluster_extended")["condition"]
    .mean()
    * 100
)


force78_rate = (
    B.assign(
        condition=(
            (B["Force"] >= 7) &
            (B["Force"] < 9)
        )
    )
    .groupby("Cluster_extended")["condition"]
    .mean()
    * 100
)


summary["Force_2xxx_%"] = force2_rate
summary["Force_7_8_%"] = force78_rate


print("\n" + "=" * 100)
print("Cluster별 공정 특성")
print("=" * 100)

print(
    summary.round(4).to_string()
)


# ============================================================
# 7. 날짜별 Cluster 개수
# ============================================================

date_count = pd.crosstab(
    B["Date"],
    B["Cluster_extended"]
)


print("\n" + "=" * 100)
print("B 날짜별 Cluster 개수")
print("=" * 100)

print(
    date_count.to_string()
)


# ============================================================
# 8. 날짜별 Cluster 비율
# ============================================================

date_rate = (
    pd.crosstab(
        B["Date"],
        B["Cluster_extended"],
        normalize="index"
    )
    * 100
)


print("\n" + "=" * 100)
print("B 날짜별 Cluster 비율 (%)")
print("=" * 100)

print(
    date_rate
    .round(2)
    .to_string()
)


# ============================================================
# 9. 날짜마다 정확히 같은 개수인 Cluster 찾기
# ============================================================

same_count_clusters = []


for c in date_count.columns:

    values = date_count[c].values

    if len(set(values)) == 1:

        same_count_clusters.append(
            (
                c,
                int(values[0])
            )
        )


print("\n" + "=" * 70)
print("4개 B 날짜에서 정확히 같은 개수로 반복되는 Cluster")
print("=" * 70)


if len(same_count_clusters) == 0:

    print("없음")

else:

    for c, n in same_count_clusters:

        print(
            f"Cluster {c}: "
            f"각 날짜 {n}개씩"
        )


# ============================================================
# 10. 저장
# ============================================================

result_df.to_csv(
    "B_kmeans_K2_K12_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    "B_kmeans_extended_summary.csv",
    encoding="utf-8-sig"
)

date_count.to_csv(
    "B_kmeans_extended_date_count.csv",
    encoding="utf-8-sig"
)


print("\n" + "=" * 70)
print("확인할 것")
print("=" * 70)

print("""
1. Silhouette가 어느 K에서 최고인지

2. K가 커질수록 계속 상승하는지,
   어느 지점부터 떨어지는지

3. 최고 K에서 각 Cluster가
   어떤 Force / Current / Voltage / Weld Time 특징인지

4. 4개 B 날짜에서
   정확히 같은 개수로 반복되는 Cluster가 있는지
""")

B공정 K-Means K=2~12 추가 탐색
K= 2 | Silhouette=0.4525 | 최소 군집=1116 | 최대 군집=4484
K= 3 | Silhouette=0.4121 | 최소 군집= 956 | 최대 군집=3032
K= 4 | Silhouette=0.4469 | 최소 군집= 280 | 최대 군집=2944
K= 5 | Silhouette=0.4830 | 최소 군집= 280 | 최대 군집=2351
K= 6 | Silhouette=0.4927 | 최소 군집=  72 | 최대 군집=2351
K= 7 | Silhouette=0.5082 | 최소 군집=  72 | 최대 군집=2115
K= 8 | Silhouette=0.5443 | 최소 군집=  72 | 최대 군집=1902
K= 9 | Silhouette=0.5510 | 최소 군집=  72 | 최대 군집=1886
K=10 | Silhouette=0.5830 | 최소 군집=  72 | 최대 군집=1886
K=11 | Silhouette=0.5887 | 최소 군집=  72 | 최대 군집=1886
K=12 | Silhouette=0.5914 | 최소 군집=  72 | 최대 군집=1890

Silhouette 순위
 K  Silhouette  Smallest_Cluster  Largest_Cluster
12    0.591392                72             1890
11    0.588687                72             1886
10    0.583043                72             1886
 9    0.551023                72             1886
 8    0.544332                72             1902
 7    0.508218                72             2115
 6    0.492730                72             2351

In [6]:
# ============================================================
# B공정을 Force 조건별로 나눈 뒤 내부 패턴 탐색
#
# 1) B공정 → Force 2.xxx / Force 7~8 분리
# 2) 각각 Current / Voltage / Weld Time만 사용
# 3) K=2~10 비교
# 4) Silhouette Score 확인
# 5) 선택된 K의 Cluster 특성 확인
# 6) 4개 B 날짜에서 반복되는지 확인
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# ============================================================
# 1. B 데이터 확인
# ============================================================

required = ["Force", "Current", "Voltage", "Weld_Time", "Date"]

missing = [c for c in required if c not in B.columns]

if missing:
    raise ValueError(f"B 데이터에 필요한 컬럼이 없습니다: {missing}")


# ============================================================
# 2. Force 기준으로 분리
# ============================================================

B_force2 = B[
    (B["Force"] >= 2) &
    (B["Force"] < 3)
].copy()


B_force78 = B[
    (B["Force"] >= 7) &
    (B["Force"] < 9)
].copy()


print("=" * 80)
print("B공정 Force 조건 분리")
print("=" * 80)

print("B 전체:", len(B))
print("Force 2.xxx:", len(B_force2))
print("Force 7~8:", len(B_force78))
print("그 외:", len(B) - len(B_force2) - len(B_force78))


# ============================================================
# 3. 분석 함수
# ============================================================

def analyze_force_group(df_group, group_name):

    print("\n\n")
    print("=" * 90)
    print(group_name)
    print("=" * 90)

    print("전체 행:", len(df_group))

    print("\n[날짜별 행 개수]")

    date_original_count = (
        df_group["Date"]
        .value_counts()
        .sort_index()
    )

    print(date_original_count.to_string())


    # --------------------------------------------------------
    # K-Means 입력
    #
    # Force는 이미 조건을 나누는 데 사용했으므로 제외
    # Date도 제외
    # --------------------------------------------------------

    features = [
        "Current",
        "Voltage",
        "Weld_Time"
    ]

    X = df_group[features].copy()

    X = X.dropna()

    # 인덱스 맞추기
    work = df_group.loc[X.index].copy()


    # --------------------------------------------------------
    # 표준화
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)


    # --------------------------------------------------------
    # K=2~10 비교
    # --------------------------------------------------------

    results = []

    models = {}

    print("\n" + "-" * 70)
    print("K별 Silhouette Score")
    print("-" * 70)


    max_k = min(
        10,
        len(X) - 1
    )


    for k in range(2, max_k + 1):

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=50
        )

        labels = model.fit_predict(X_scaled)

        score = silhouette_score(
            X_scaled,
            labels
        )

        counts = pd.Series(labels).value_counts()

        results.append({
            "K": k,
            "Silhouette": score,
            "Smallest_Cluster": counts.min(),
            "Largest_Cluster": counts.max()
        })

        models[k] = model

        print(
            f"K={k:2d}"
            f" | Silhouette={score:.4f}"
            f" | 최소={counts.min():4d}"
            f" | 최대={counts.max():4d}"
        )


    result_df = pd.DataFrame(results)


    # --------------------------------------------------------
    # 현재 탐색범위 최고 K
    # --------------------------------------------------------

    best_row = result_df.loc[
        result_df["Silhouette"].idxmax()
    ]

    best_k = int(
        best_row["K"]
    )

    best_score = float(
        best_row["Silhouette"]
    )


    print("\n" + "-" * 70)
    print("현재 탐색범위 최고 결과")
    print("-" * 70)

    print("K =", best_k)

    print(
        "Silhouette =",
        round(best_score, 4)
    )


    # --------------------------------------------------------
    # 최고 K 적용
    # --------------------------------------------------------

    final_model = models[best_k]

    work["Cluster"] = final_model.labels_


    # --------------------------------------------------------
    # Cluster별 개수
    # --------------------------------------------------------

    cluster_count = (
        work["Cluster"]
        .value_counts()
        .sort_index()
    )


    print("\n" + "-" * 70)
    print("Cluster별 개수")
    print("-" * 70)


    for cluster, count in cluster_count.items():

        print(
            f"Cluster {cluster}: "
            f"{count}개 "
            f"({count / len(work) * 100:.2f}%)"
        )


    # --------------------------------------------------------
    # Cluster별 센서 특징
    # --------------------------------------------------------

    summary = (
        work
        .groupby("Cluster")
        .agg(

            Count=("Current", "size"),

            Current_mean=("Current", "mean"),
            Current_std=("Current", "std"),

            Voltage_mean=("Voltage", "mean"),
            Voltage_std=("Voltage", "std"),

            WeldTime_mean=("Weld_Time", "mean"),
            WeldTime_std=("Weld_Time", "std"),

            # 참고용
            Force_mean=("Force", "mean"),
            Force_std=("Force", "std")
        )
    )


    print("\n" + "-" * 90)
    print("Cluster별 공정 특성")
    print("-" * 90)

    print(
        summary
        .round(4)
        .to_string()
    )


    # --------------------------------------------------------
    # 날짜 × Cluster 개수
    # --------------------------------------------------------

    date_count = pd.crosstab(
        work["Date"],
        work["Cluster"]
    )


    print("\n" + "-" * 90)
    print("날짜별 Cluster 개수")
    print("-" * 90)

    print(
        date_count.to_string()
    )


    # --------------------------------------------------------
    # 날짜 × Cluster 비율
    # --------------------------------------------------------

    date_rate = (
        pd.crosstab(
            work["Date"],
            work["Cluster"],
            normalize="index"
        )
        * 100
    )


    print("\n" + "-" * 90)
    print("날짜별 Cluster 비율 (%)")
    print("-" * 90)

    print(
        date_rate
        .round(2)
        .to_string()
    )


    # --------------------------------------------------------
    # 4개 날짜 모두 존재하는 Cluster
    # --------------------------------------------------------

    all_dates = sorted(
        work["Date"].unique()
    )

    print("\n" + "-" * 90)
    print("4개 B 날짜 반복 여부")
    print("-" * 90)


    for cluster in sorted(work["Cluster"].unique()):

        cluster_dates = sorted(
            work.loc[
                work["Cluster"] == cluster,
                "Date"
            ].unique()
        )

        print(
            f"Cluster {cluster}: "
            f"{len(cluster_dates)}/{len(all_dates)} 날짜"
        )


    # --------------------------------------------------------
    # 날짜마다 정확히 같은 개수인지
    # --------------------------------------------------------

    print("\n" + "-" * 90)
    print("4개 날짜에서 정확히 같은 개수로 반복되는 Cluster")
    print("-" * 90)

    same_clusters = []


    for cluster in date_count.columns:

        # 모든 날짜가 표에 있는지
        counts = date_count[cluster].values

        if len(counts) == 4 and len(set(counts)) == 1:

            same_clusters.append(
                (
                    cluster,
                    int(counts[0])
                )
            )


    if len(same_clusters) == 0:

        print("없음")

    else:

        for cluster, count in same_clusters:

            print(
                f"Cluster {cluster}: "
                f"각 날짜 {count}개씩"
            )


    # --------------------------------------------------------
    # 저장
    # --------------------------------------------------------

    safe_name = (
        group_name
        .replace(" ", "_")
        .replace(".", "")
        .replace("~", "_")
    )


    work.to_csv(
        f"{safe_name}_kmeans_rows.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result_df.to_csv(
        f"{safe_name}_k_scores.csv",
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        f"{safe_name}_cluster_summary.csv",
        encoding="utf-8-sig"
    )


    return {
        "name": group_name,
        "best_k": best_k,
        "best_score": best_score,
        "scores": result_df,
        "summary": summary,
        "date_count": date_count,
        "date_rate": date_rate,
        "data": work
    }


# ============================================================
# 4. Force 2.xxx 분석
# ============================================================

result_force2 = analyze_force_group(
    B_force2,
    "B_Force_2.xxx"
)


# ============================================================
# 5. Force 7~8 분석
# ============================================================

result_force78 = analyze_force_group(
    B_force78,
    "B_Force_7~8"
)


# ============================================================
# 6. 최종 비교
# ============================================================

print("\n\n")
print("=" * 90)
print("최종 비교")
print("=" * 90)

print(
    "Force 2.xxx"
    f" → 현재 최고 K={result_force2['best_k']}"
    f" | Silhouette={result_force2['best_score']:.4f}"
)

print(
    "Force 7~8"
    f" → 현재 최고 K={result_force78['best_k']}"
    f" | Silhouette={result_force78['best_score']:.4f}"
)


print("""
해석할 때 볼 것

[Force 2.xxx]
Current / Voltage / Weld Time만으로
서로 다른 반복 패턴이 존재하는가?

[Force 7~8]
Current / Voltage / Weld Time만으로
서로 다른 반복 패턴이 존재하는가?

중요:
K가 여러 개 나온다고 실제 제조공정이 그 개수만큼
존재한다고 바로 해석하지 않는다.

1. Silhouette
2. Cluster별 센서값 차이
3. 4개 B 날짜에서 반복되는지
4. 날짜별 개수/비율이 비슷한지

이 네 가지를 같이 본다.
""")

B공정 Force 조건 분리
B 전체: 5600
Force 2.xxx: 4428
Force 7~8: 616
그 외: 556



B_Force_2.xxx
전체 행: 4428

[날짜별 행 개수]
Date
03/25    1059
03/27    1355
03/31    1507
04/03     507

----------------------------------------------------------------------
K별 Silhouette Score
----------------------------------------------------------------------
K= 2 | Silhouette=0.4141 | 최소=1500 | 최대=2928
K= 3 | Silhouette=0.4239 | 최소=  16 | 최대=2920
K= 4 | Silhouette=0.4841 | 최소=  16 | 최대=2297
K= 5 | Silhouette=0.5100 | 최소=  16 | 최대=2139
K= 6 | Silhouette=0.5394 | 최소=  16 | 최대=1882
K= 7 | Silhouette=0.5708 | 최소=  16 | 최대=1882
K= 8 | Silhouette=0.5771 | 최소=  16 | 최대=1882
K= 9 | Silhouette=0.5687 | 최소=  16 | 최대=1822
K=10 | Silhouette=0.5833 | 최소=  16 | 최대=1822

----------------------------------------------------------------------
현재 탐색범위 최고 결과
----------------------------------------------------------------------
K = 10
Silhouette = 0.5833

----------------------------------------------------------------------
Cluste

In [7]:
# ============================================================
# A Force 2.xxx vs B Force 2.xxx 비교
#
# 목적
# 1. A/B 모두 Force 2.xxx 행만 추출
# 2. Current / Voltage / Weld Time 비교
# 3. A/B 라벨을 넣지 않고 K-Means K=2
# 4. 실제 A/B와 K-Means 결과 비교
#
# 그래프 없음
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# ============================================================
# 1. 원본 파일
# ============================================================

file_path = r"c:\Kamp-2026\experiments\YSJ\YSJ-002\Welding Data Set_01.xlsx"

raw = pd.read_excel(
    file_path,
    sheet_name="Raw data"
)


# ============================================================
# 2. 분석용 데이터
# ============================================================

df = pd.DataFrame({

    "Force": pd.to_numeric(
        raw["weld force(bar)"],
        errors="coerce"
    ),

    "Current": pd.to_numeric(
        raw["weld current(kA)"],
        errors="coerce"
    ),

    "Voltage": pd.to_numeric(
        raw["weld Voltage(v)"],
        errors="coerce"
    ),

    "Weld_Time": pd.to_numeric(
        raw["weld time(ms)"],
        errors="coerce"
    ),

    "Date": pd.to_datetime(
        raw["working time"],
        errors="coerce"
    ).dt.strftime("%m/%d")
})


df = df.dropna().reset_index(drop=True)


# ============================================================
# 3. A / B 날짜
# ============================================================

A_dates = [
    "03/24",
    "03/26",
    "03/30",
    "04/02",
    "04/07"
]

B_dates = [
    "03/25",
    "03/27",
    "03/31",
    "04/03"
]


df["Process"] = np.where(
    df["Date"].isin(A_dates),
    "A",
    np.where(
        df["Date"].isin(B_dates),
        "B",
        "Unknown"
    )
)


df = df[
    df["Process"] != "Unknown"
].copy()


# ============================================================
# 4. Force 2.xxx만 추출
# ============================================================

force2 = df[
    (df["Force"] >= 2) &
    (df["Force"] < 3)
].copy()


A2 = force2[
    force2["Process"] == "A"
].copy()

B2 = force2[
    force2["Process"] == "B"
].copy()


print("=" * 80)
print("A Force 2.xxx vs B Force 2.xxx")
print("=" * 80)

print("A Force 2.xxx:", len(A2), "행")
print("B Force 2.xxx:", len(B2), "행")


# ============================================================
# 5. 날짜별 행 개수
# ============================================================

print("\n" + "=" * 80)
print("날짜별 Force 2.xxx 행 개수")
print("=" * 80)

print(
    force2.groupby(
        ["Process", "Date"]
    )
    .size()
    .to_string()
)


# ============================================================
# 6. A/B 기본 통계 비교
# ============================================================

features = [
    "Current",
    "Voltage",
    "Weld_Time"
]


summary = (
    force2
    .groupby("Process")[features]
    .agg(["mean", "std", "median", "min", "max"])
)


print("\n" + "=" * 100)
print("A/B Force 2.xxx 공정값 비교")
print("=" * 100)

print(
    summary.round(4).to_string()
)


# ============================================================
# 7. 평균 차이
# ============================================================

A_mean = A2[features].mean()
B_mean = B2[features].mean()


print("\n" + "=" * 80)
print("B - A 평균 차이")
print("=" * 80)


for feature in features:

    diff = (
        B_mean[feature]
        -
        A_mean[feature]
    )

    print(
        f"{feature}: "
        f"A={A_mean[feature]:.4f}"
        f" | B={B_mean[feature]:.4f}"
        f" | B-A={diff:+.4f}"
    )


# ============================================================
# 8. K-Means
#
# 중요:
# Process(A/B), Date, Force 사용 X
#
# Current / Voltage / Weld Time만 사용
# ============================================================

X = force2[
    [
        "Current",
        "Voltage",
        "Weld_Time"
    ]
].copy()


scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


model = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=50
)


force2["Cluster"] = model.fit_predict(
    X_scaled
)


score = silhouette_score(
    X_scaled,
    force2["Cluster"]
)


# ============================================================
# 9. K-Means 개수
# ============================================================

print("\n" + "=" * 80)
print("K-Means 결과")
print("=" * 80)

print(
    "Silhouette Score:",
    round(score, 4)
)


print("\nCluster 개수")

print(
    force2["Cluster"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 10. Cluster 특징
# ============================================================

cluster_summary = (
    force2
    .groupby("Cluster")[features]
    .agg(["count", "mean", "std"])
)


print("\n" + "=" * 100)
print("Cluster별 공정 특성")
print("=" * 100)

print(
    cluster_summary
    .round(4)
    .to_string()
)


# ============================================================
# 11. 실제 A/B × K-Means
# ============================================================

cross_count = pd.crosstab(
    force2["Process"],
    force2["Cluster"]
)


cross_rate = (
    pd.crosstab(
        force2["Process"],
        force2["Cluster"],
        normalize="index"
    )
    * 100
)


print("\n" + "=" * 80)
print("실제 A/B × K-Means 개수")
print("=" * 80)

print(
    cross_count.to_string()
)


print("\n" + "=" * 80)
print("실제 A/B × K-Means 비율 (%)")
print("=" * 80)

print(
    cross_rate
    .round(2)
    .to_string()
)


# ============================================================
# 12. 날짜별 Cluster 비율
# ============================================================

date_rate = (
    pd.crosstab(
        force2["Date"],
        force2["Cluster"],
        normalize="index"
    )
    * 100
)


print("\n" + "=" * 80)
print("날짜별 Cluster 비율 (%)")
print("=" * 80)

print(
    date_rate
    .round(2)
    .to_string()
)


# ============================================================
# 13. 날짜별 Current / Voltage / Weld Time 평균
# ============================================================

daily_mean = (
    force2
    .groupby(["Process", "Date"])[features]
    .mean()
)


print("\n" + "=" * 100)
print("날짜별 Force 2.xxx 평균")
print("=" * 100)

print(
    daily_mean
    .round(4)
    .to_string()
)


# ============================================================
# 14. 저장
# ============================================================

force2.to_csv(
    "A_vs_B_Force2xxx_kmeans.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 80)
print("분석 완료")
print("=" * 80)

print("""
핵심적으로 볼 결과

1. B - A 평균 차이
   → 같은 Force 2.xxx에서도
     Current / Voltage / Weld Time이 다른가?

2. 실제 A/B × K-Means 비율
   → A와 B가 서로 다른 Cluster에 몰리는가?

3. 날짜별 Cluster 비율
   → 특정 날짜 하나 때문에 생긴 차이가 아닌가?

해석:

A/B가 서로 다른 Cluster에 강하게 몰리면
→ Force 7~8의 존재만이 A/B 차이가 아님
→ Force 2.xxx 상태 자체도 A/B에서 다름

A/B의 Cluster 비율이 비슷하면
→ A의 기본공정과 B의 Force 2.xxx 공정은 유사
→ B의 핵심적인 추가 특징은 Force 7~8 구간일 가능성이 커짐
""")

A Force 2.xxx vs B Force 2.xxx
A Force 2.xxx: 6339 행
B Force 2.xxx: 4428 행

날짜별 Force 2.xxx 행 개수
Process  Date 
A        03/24    1200
         03/26    1000
         03/30    1470
         04/02    2000
         04/07     669
B        03/25    1059
         03/27    1355
         03/31    1507
         04/03     507

A/B Force 2.xxx 공정값 비교
         Current                              Voltage                              Weld_Time                           
            mean     std median    min    max    mean     std median    min    max      mean     std median   min   max
Process                                                                                                                
A        14.6724  0.0764  14.71  14.52  14.97   2.702  0.0042  2.701  2.689  2.717   71.7295  0.6307   72.0  70.0  73.0
B        14.7324  0.0942  14.74  14.52  15.07   2.702  0.0131  2.702  2.481  2.761   71.7202  0.6358   72.0  70.0  73.0

B - A 평균 차이
Current: A=14.6724 | B=14.7324 | B-A=+0.0600

In [10]:
import pandas as pd
from pathlib import Path

# 현재 위치: experiments/YSJ/YSJ-004
BASE = Path.cwd()

print("현재 위치:", BASE)

# 이전 실험(YSJ-003)의 CSV 확인
AE_FOLDER = BASE.parent / "YSJ-003"

print("\nYSJ-003 CSV 목록")
for f in AE_FOLDER.glob("*.csv"):
    print(f.name)

현재 위치: c:\Kamp-2026\experiments\YSJ\YSJ-004

YSJ-003 CSV 목록
AE_3cluster_result.csv
AE_3cluster_summary.csv
AE_actual_direction_summary.csv
AE_C0_baseline.csv
AE_C1_C2_date_row_detail.csv
AE_C1_C2_dominant_feature_count.csv
AE_C1_C2_dominant_feature_rate.csv
AE_C1_C2_error_type_detail.csv
AE_C1_C2_feature_analysis.csv
AE_C1_C2_feature_error.csv
AE_C2_detail.csv
AE_candidates_after_B_pattern.csv
AE_candidates_with_B_pattern.csv
AE_candidate_actual_direction_detail.csv
AE_cluster_direction_pattern.csv
AE_cluster_error_type_count.csv
AE_cluster_error_type_rate.csv
AE_cluster_feature_mean_error.csv
AE_continuous_blocks.csv
AE_daily_candidate_analysis.csv
AE_error_3cluster_result.csv
AE_error_type_actual_values.csv
AE_error_type_summary.csv
AE_extreme_pattern_count.csv
AE_force_condition_summary.csv
AE_force_group_comparison.csv
AE_force_pattern_detail.csv
AE_force_pattern_rate.csv
AE_pattern_by_force_count.csv
AE_pattern_by_force_rate.csv
AE_positions_repeated_4dates.csv
AE_repeated_blocks.

In [11]:
import pandas as pd
from pathlib import Path

# YSJ-003의 기존 AE C1/C2 결과
file_path = Path.cwd().parent / "YSJ-003" / "AE_C1_C2_date_row_detail.csv"

ae = pd.read_csv(file_path)

print("=" * 70)
print("AE C1/C2 파일 확인")
print("=" * 70)

print("행 개수:", len(ae))

print("\n컬럼:")
for col in ae.columns:
    print("-", col)

print("\n앞 5행:")
print(ae.head().to_string())

AE C1/C2 파일 확인
행 개수: 1061

컬럼:
- Original_Row
- Date
- Working_Time
- AE_Cluster
- AE_Type
- Force
- Current
- Voltage
- Weld_Time
- Reconstruction_Error
- Current_Error
- Voltage_Error
- Weld_Time_Error

앞 5행:
   Original_Row        Date Working_Time  AE_Cluster     AE_Type  Force  Current  Voltage  Weld_Time  Reconstruction_Error  Current_Error  Voltage_Error  Weld_Time_Error
0         10884  2020-04-03   2020-04-03           2  C2_Voltage   9.01    14.83    2.464       71.0             28.117209       1.457217      91.746664         1.302161
1          7808  2020-03-31   2020-03-31           2  C2_Voltage   9.01    14.83    2.464       71.0             28.117209       1.457217      91.746664         1.302161
2          1884  2020-03-25   2020-03-25           2  C2_Voltage   9.01    14.83    2.464       71.0             28.117209       1.457217      91.746664         1.302161
3          4698  2020-03-27   2020-03-27           2  C2_Voltage   9.01    14.83    2.464       71.0         

In [12]:
# ============================================================
# 기존 AE 후보 1,061개 ↔ A/B 공정구조 연결
# ============================================================

import pandas as pd
from pathlib import Path

BASE = Path.cwd()

# ------------------------------------------------------------
# 1. 원본 데이터
# ------------------------------------------------------------

raw_path = BASE.parent / "YSJ-002" / "Welding Data Set_01.xlsx"
ae_path  = BASE.parent / "YSJ-003" / "AE_C1_C2_date_row_detail.csv"

raw = pd.read_excel(raw_path, sheet_name="Raw data")
ae = pd.read_csv(ae_path)

# 원본 행 번호 생성
raw = raw.reset_index(drop=True)
raw["Original_Row"] = raw.index

# 컬럼 정리
raw["Force"] = pd.to_numeric(raw["weld force(bar)"], errors="coerce")

raw["Date"] = pd.to_datetime(
    raw["working time"],
    errors="coerce"
).dt.strftime("%m/%d")


# ------------------------------------------------------------
# 2. A/B 날짜
# ------------------------------------------------------------

A_dates = ["03/24", "03/26", "03/30", "04/02", "04/07"]
B_dates = ["03/25", "03/27", "03/31", "04/03"]


# ------------------------------------------------------------
# 3. 공정구간 분류
# ------------------------------------------------------------

def classify_group(row):

    date = row["Date"]
    force = row["Force"]

    if date in A_dates and 2 <= force < 3:
        return "A_Force_2xxx"

    elif date in B_dates and 2 <= force < 3:
        return "B_Force_2xxx"

    elif date in B_dates and 7 <= force < 9:
        return "B_Force_7_8"

    else:
        return "Other"


raw["Process_Group"] = raw.apply(
    classify_group,
    axis=1
)


# ------------------------------------------------------------
# 4. 기존 AE 후보 표시
# ------------------------------------------------------------

ae_rows = set(ae["Original_Row"])

raw["AE_Candidate"] = raw["Original_Row"].isin(ae_rows)


# ------------------------------------------------------------
# 5. 세 공정구간 비교
# ------------------------------------------------------------

groups = [
    "A_Force_2xxx",
    "B_Force_2xxx",
    "B_Force_7_8"
]

result = []

for group in groups:

    temp = raw[raw["Process_Group"] == group]

    total = len(temp)
    candidate = int(temp["AE_Candidate"].sum())

    rate = candidate / total * 100 if total > 0 else 0

    result.append({
        "공정구간": group,
        "전체행": total,
        "AE후보": candidate,
        "AE후보율(%)": round(rate, 2)
    })


result = pd.DataFrame(result)


print("=" * 75)
print("공정구간별 기존 AE 후보 비교")
print("=" * 75)

print(result.to_string(index=False))


# ------------------------------------------------------------
# 6. AE 후보 1,061개 중 각 구간에 몇 개가 있는지도 확인
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("AE 후보 1,061개 분포")
print("=" * 75)

ae_distribution = (
    raw[raw["AE_Candidate"]]
    ["Process_Group"]
    .value_counts()
)

print(ae_distribution.to_string())


# ------------------------------------------------------------
# 7. B Force 7~8 날짜별 확인
# ------------------------------------------------------------

b78 = raw[
    raw["Process_Group"] == "B_Force_7_8"
]

b78_daily = (
    b78.groupby("Date")
    .agg(
        전체행=("AE_Candidate", "size"),
        AE후보=("AE_Candidate", "sum")
    )
)

b78_daily["AE후보율(%)"] = (
    b78_daily["AE후보"]
    / b78_daily["전체행"]
    * 100
).round(2)


print("\n" + "=" * 75)
print("B Force 7~8 날짜별 AE 후보")
print("=" * 75)

print(b78_daily.to_string())


print("\n" + "=" * 75)
print("확인할 것")
print("=" * 75)

print("""
A Force 2.xxx
B Force 2.xxx
B Force 7~8

→ 세 구간 중 어디에서 AE 후보율이 높은지만 비교.

특히 B Force 7~8의 AE 후보율이
나머지 두 구간보다 확실히 높은지 확인.
""")

공정구간별 기존 AE 후보 비교
        공정구간  전체행  AE후보  AE후보율(%)
A_Force_2xxx 6339     1      0.02
B_Force_2xxx 4428    56      1.26
 B_Force_7_8  616   616    100.00

AE 후보 1,061개 분포
Process_Group
B_Force_7_8     616
Other           388
B_Force_2xxx     56
A_Force_2xxx      1

B Force 7~8 날짜별 AE 후보
       전체행  AE후보  AE후보율(%)
Date                      
03/25  154   154     100.0
03/27  154   154     100.0
03/31  154   154     100.0
04/03  154   154     100.0

확인할 것

A Force 2.xxx
B Force 2.xxx
B Force 7~8

→ 세 구간 중 어디에서 AE 후보율이 높은지만 비교.

특히 B Force 7~8의 AE 후보율이
나머지 두 구간보다 확실히 높은지 확인.



In [14]:
# ============================================================
# AE 후보 1,061개 중 Other 388개 확인
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. 파일 불러오기
# ------------------------------------------------------------

BASE = Path.cwd()

raw_path = BASE.parent / "YSJ-002" / "Welding Data Set_01.xlsx"
ae_path  = BASE.parent / "YSJ-003" / "AE_C1_C2_date_row_detail.csv"

raw = pd.read_excel(raw_path, sheet_name="Raw data")
ae = pd.read_csv(ae_path)

raw = raw.reset_index(drop=True)
raw["Original_Row"] = raw.index

# 필요한 컬럼만 새로 생성
raw["Force"] = pd.to_numeric(raw["weld force(bar)"], errors="coerce")
raw["Current"] = pd.to_numeric(raw["weld current(kA)"], errors="coerce")
raw["Voltage"] = pd.to_numeric(raw["weld Voltage(v)"], errors="coerce")
raw["Weld_Time"] = pd.to_numeric(raw["weld time(ms)"], errors="coerce")

raw["Date"] = pd.to_datetime(
    raw["working time"],
    errors="coerce"
).dt.strftime("%m/%d")


# ------------------------------------------------------------
# 2. A/B 날짜
# ------------------------------------------------------------

A_dates = ["03/24", "03/26", "03/30", "04/02", "04/07"]
B_dates = ["03/25", "03/27", "03/31", "04/03"]


# ------------------------------------------------------------
# 3. 공정 그룹 분류
# ------------------------------------------------------------

def process_group(row):

    date = row["Date"]
    force = row["Force"]

    if date in A_dates and 2 <= force < 3:
        return "A_Force_2xxx"

    if date in B_dates and 2 <= force < 3:
        return "B_Force_2xxx"

    if date in B_dates and 7 <= force < 9:
        return "B_Force_7_8"

    return "Other"


raw["Process_Group"] = raw.apply(process_group, axis=1)


# ------------------------------------------------------------
# 4. AE 후보 표시
# ------------------------------------------------------------

ae_rows = set(ae["Original_Row"])

raw["AE_Candidate"] = raw["Original_Row"].isin(ae_rows)


# ------------------------------------------------------------
# 5. Other AE 후보 추출
# ------------------------------------------------------------

other = raw[
    (raw["AE_Candidate"] == True) &
    (raw["Process_Group"] == "Other")
].copy()


print("=" * 70)
print("AE 후보 Other 분석")
print("=" * 70)

print("Other 개수:", len(other))


# ------------------------------------------------------------
# 6. 날짜별 개수
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("날짜별 Other 개수")
print("=" * 70)

print(
    other["Date"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 7. Force 범위별 개수
# ------------------------------------------------------------

def force_range(x):

    if 1 <= x < 2:
        return "1.xxx"

    elif 2 <= x < 3:
        return "2.xxx"

    elif 3 <= x < 4:
        return "3.xxx"

    elif 4 <= x < 5:
        return "4.xxx"

    elif 5 <= x < 6:
        return "5.xxx"

    elif 6 <= x < 7:
        return "6.xxx"

    elif 7 <= x < 8:
        return "7.xxx"

    elif 8 <= x < 9:
        return "8.xxx"

    elif 9 <= x < 10:
        return "9.xxx"

    elif 10 <= x < 11:
        return "10.xxx"

    else:
        return "기타"


other["Force_Range"] = other["Force"].apply(force_range)


print("\n" + "=" * 70)
print("Force 범위별 개수")
print("=" * 70)

print(other["Force_Range"].value_counts())


# ------------------------------------------------------------
# 8. 실제 Force 값 TOP 20
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("Force 값 TOP 20")
print("=" * 70)

print(
    other["Force"]
    .value_counts()
    .head(20)
)


# ------------------------------------------------------------
# 9. 날짜 × Force 범위
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("날짜 × Force 범위")
print("=" * 70)

table = pd.crosstab(
    other["Date"],
    other["Force_Range"]
)

print(table)


# ------------------------------------------------------------
# 10. 전체 센서값 평균/최소/최대
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("Other 센서값")
print("=" * 70)

for col in ["Force", "Current", "Voltage", "Weld_Time"]:

    print(f"\n[{col}]")
    print("개수 :", other[col].count())
    print("평균 :", round(other[col].mean(), 4))
    print("표준편차 :", round(other[col].std(), 4))
    print("최소 :", round(other[col].min(), 4))
    print("중앙값 :", round(other[col].median(), 4))
    print("최대 :", round(other[col].max(), 4))


# ------------------------------------------------------------
# 11. AE 파일과 연결
# ------------------------------------------------------------

other_detail = other.merge(
    ae[
        [
            "Original_Row",
            "AE_Cluster",
            "AE_Type",
            "Reconstruction_Error",
            "Current_Error",
            "Voltage_Error",
            "Weld_Time_Error"
        ]
    ],
    on="Original_Row",
    how="left"
)


# ------------------------------------------------------------
# 12. AE Type 개수
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("Other의 AE Type")
print("=" * 70)

print(
    other_detail["AE_Type"]
    .value_counts()
)


# ------------------------------------------------------------
# 13. AE Cluster 개수
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("Other의 AE Cluster")
print("=" * 70)

print(
    other_detail["AE_Cluster"]
    .value_counts()
)


print("\n" + "=" * 70)
print("분석 완료")
print("=" * 70)

print("""
볼 것은 4개만:

1. 388개가 어느 날짜에 있는가
2. Force가 어느 범위에 몰려 있는가
3. 특정 Force 값이 반복되는가
4. AE_Type이 무엇인가
""")

AE 후보 Other 분석
Other 개수: 388

날짜별 Other 개수
Date
03/25    97
03/27    97
03/31    97
04/03    97
Name: count, dtype: int64

Force 범위별 개수
Force_Range
6.xxx     208
9.xxx      48
4.xxx      48
3.xxx      44
5.xxx      32
10.xxx      8
Name: count, dtype: int64

Force 값 TOP 20
Force
9.27     12
6.70     12
6.74     12
6.78     12
6.94     12
6.85     12
6.83     12
5.14     12
10.54     8
9.07      8
9.01      8
9.05      8
9.04      8
6.64      8
6.63      8
6.67      8
6.82      8
6.91      8
6.88      8
6.95      8
Name: count, dtype: int64

날짜 × Force 범위
Force_Range  10.xxx  3.xxx  4.xxx  5.xxx  6.xxx  9.xxx
Date                                                  
03/25             2     11     12      8     52     12
03/27             2     11     12      8     52     12
03/31             2     11     12      8     52     12
04/03             2     11     12      8     52     12

Other 센서값

[Force]
개수 : 388
평균 : 6.3479
표준편차 : 1.6889
최소 : 3.01
중앙값 : 6.7
최대 : 10.54

[Current]
개수 : 388
평균 

In [15]:
# ============================================================
# B공정의 AE 반복구간이 하나의 연속 공정 시퀀스인지 확인
#
# 확인 대상
# - B Force 7~8 : 154개 × 4일
# - Other       :  97개 × 4일
#
# 목표:
# 원본 행 순서에서 이 251개가 서로 연결되어 반복되는지 확인
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. 파일 불러오기
# ------------------------------------------------------------

BASE = Path.cwd()

raw_path = BASE.parent / "YSJ-002" / "Welding Data Set_01.xlsx"
ae_path  = BASE.parent / "YSJ-003" / "AE_C1_C2_date_row_detail.csv"

raw = pd.read_excel(raw_path, sheet_name="Raw data")
ae = pd.read_csv(ae_path)

raw = raw.reset_index(drop=True)
raw["Original_Row"] = raw.index

raw["Force"] = pd.to_numeric(
    raw["weld force(bar)"],
    errors="coerce"
)

raw["Current"] = pd.to_numeric(
    raw["weld current(kA)"],
    errors="coerce"
)

raw["Voltage"] = pd.to_numeric(
    raw["weld Voltage(v)"],
    errors="coerce"
)

raw["Weld_Time"] = pd.to_numeric(
    raw["weld time(ms)"],
    errors="coerce"
)

raw["Date"] = pd.to_datetime(
    raw["working time"],
    errors="coerce"
).dt.strftime("%m/%d")


# ------------------------------------------------------------
# 2. B 날짜
# ------------------------------------------------------------

B_dates = [
    "03/25",
    "03/27",
    "03/31",
    "04/03"
]


# ------------------------------------------------------------
# 3. 기존 AE 후보 표시
# ------------------------------------------------------------

ae_rows = set(ae["Original_Row"])

raw["AE_Candidate"] = raw["Original_Row"].isin(ae_rows)


# ------------------------------------------------------------
# 4. 우리가 확인할 AE 반복구간 분류
# ------------------------------------------------------------

def classify(row):

    if row["Date"] not in B_dates:
        return "Not_B"

    if not row["AE_Candidate"]:
        return "Not_AE"

    f = row["Force"]

    # 기존 Force 7~8 그룹
    if 7 <= f < 9:
        return "Force_7_8"

    # 기존 Other 388개에 해당하는 Force 범위
    if (
        (3 <= f < 7) or
        (9 <= f < 11)
    ):
        return "Other_AE"

    return "Other"


raw["Sequence_Type"] = raw.apply(
    classify,
    axis=1
)


# ------------------------------------------------------------
# 5. B 날짜별 대상 추출
# ------------------------------------------------------------

target = raw[
    (raw["Date"].isin(B_dates)) &
    (raw["Sequence_Type"].isin(["Force_7_8", "Other_AE"]))
].copy()

target = target.sort_values("Original_Row")


print("=" * 80)
print("B공정 반복 AE 구간")
print("=" * 80)

print("전체:", len(target))

print("\n종류별:")
print(target["Sequence_Type"].value_counts())


print("\n날짜별:")
print(
    pd.crosstab(
        target["Date"],
        target["Sequence_Type"]
    )
)


# ------------------------------------------------------------
# 6. 날짜별 원본 행 위치 분석
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("날짜별 원본 행 위치 / 연속성")
print("=" * 80)


for date in B_dates:

    d = target[
        target["Date"] == date
    ].copy()

    d = d.sort_values("Original_Row")

    rows = d["Original_Row"].tolist()

    print("\n" + "-" * 70)
    print(date)
    print("-" * 70)

    print("대상 행 개수:", len(d))

    if len(rows) == 0:
        continue

    print("첫 행:", rows[0])
    print("마지막 행:", rows[-1])

    # 행 사이 간격
    d["Row_Gap"] = d["Original_Row"].diff()

    print(
        "바로 다음 행으로 이어지는 횟수:",
        int((d["Row_Gap"] == 1).sum())
    )

    print(
        "중간에 끊긴 횟수:",
        int((d["Row_Gap"] > 1).sum())
    )

    print("\n행 간격 분포:")
    print(
        d["Row_Gap"]
        .value_counts()
        .sort_index()
        .head(20)
    )


# ------------------------------------------------------------
# 7. 연속 블록 만들기
# ------------------------------------------------------------

target = target.sort_values(
    ["Date", "Original_Row"]
).copy()

target["Gap"] = (
    target
    .groupby("Date")["Original_Row"]
    .diff()
)

target["New_Block"] = (
    target["Gap"].isna() |
    (target["Gap"] != 1)
)

target["Block"] = (
    target
    .groupby("Date")["New_Block"]
    .cumsum()
)


# ------------------------------------------------------------
# 8. 연속 블록 요약
# ------------------------------------------------------------

block_summary = (
    target
    .groupby(["Date", "Block"])
    .agg(
        Start_Row=("Original_Row", "min"),
        End_Row=("Original_Row", "max"),
        Length=("Original_Row", "size"),

        Force_Min=("Force", "min"),
        Force_Max=("Force", "max"),

        Current_Mean=("Current", "mean"),
        Voltage_Mean=("Voltage", "mean"),

        First_Type=("Sequence_Type", "first"),
        Last_Type=("Sequence_Type", "last")
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("연속 블록")
print("=" * 80)

print(
    block_summary
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. 날짜별 Force 값 순서 확인
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("날짜별 Force 순서 - 처음 60개")
print("=" * 80)


for date in B_dates:

    d = target[
        target["Date"] == date
    ].sort_values("Original_Row")

    print(f"\n[{date}]")

    print(
        d[
            [
                "Original_Row",
                "Force",
                "Current",
                "Voltage",
                "Sequence_Type"
            ]
        ]
        .head(60)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 10. 네 날짜의 Force 순서가 동일한지 확인
# ------------------------------------------------------------

force_sequences = {}

for date in B_dates:

    d = target[
        target["Date"] == date
    ].sort_values("Original_Row")

    force_sequences[date] = (
        d["Force"]
        .round(4)
        .tolist()
    )


print("\n" + "=" * 80)
print("4개 B 날짜 Force 순서 동일 여부")
print("=" * 80)

base_date = B_dates[0]
base_seq = force_sequences[base_date]

for date in B_dates[1:]:

    same = (
        len(base_seq) == len(force_sequences[date])
        and base_seq == force_sequences[date]
    )

    print(
        f"{base_date} vs {date} : {same}"
    )


# ------------------------------------------------------------
# 11. 핵심 요약
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("최종 확인")
print("=" * 80)

print("""
확인할 것은 3개:

1. 각 날짜에서 대상이 251개인지
   → Force 7~8 154개 + Other 97개

2. 이 251개가 원본 행 순서에서
   하나 또는 몇 개의 연속 블록으로 이어지는지

3. 03/25, 03/27, 03/31, 04/03의
   Force 순서가 완전히 동일한지

만약 4개 날짜에서 Force 순서까지 동일하면
→ AE가 잡은 616 + 388개는
  랜덤 이상치가 아니라 B공정에서 반복되는
  동일한 공정 시퀀스일 가능성이 매우 커짐.
""")

B공정 반복 AE 구간
전체: 1004

종류별:
Sequence_Type
Force_7_8    616
Other_AE     388
Name: count, dtype: int64

날짜별:
Sequence_Type  Force_7_8  Other_AE
Date                              
03/25                154        97
03/27                154        97
03/31                154        97
04/03                154        97

날짜별 원본 행 위치 / 연속성

----------------------------------------------------------------------
03/25
----------------------------------------------------------------------
대상 행 개수: 251
첫 행: 1877
마지막 행: 2174
바로 다음 행으로 이어지는 횟수: 239
중간에 끊긴 횟수: 11

행 간격 분포:
Row_Gap
1.0     239
2.0       4
3.0       1
5.0       1
6.0       1
7.0       1
8.0       1
9.0       1
12.0      1
Name: count, dtype: int64

----------------------------------------------------------------------
03/27
----------------------------------------------------------------------
대상 행 개수: 251
첫 행: 4691
마지막 행: 4988
바로 다음 행으로 이어지는 횟수: 239
중간에 끊긴 횟수: 11

행 간격 분포:
Row_Gap
1.0     239
2.0       4
3.0       1
5.0       1
6.0

In [16]:
# ============================================================
# B공정 반복구간 전체 비교
#
# 목표
# 1. B 날짜별 AE 반복구간의 시작~끝 찾기
# 2. 그 사이의 원본 행을 전부 추출
# 3. Force / Current / Voltage / Weld Time 비교
# 4. 4개 날짜에서 동일한 반복 공정 시퀀스인지 확인
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. 데이터 불러오기
# ============================================================

BASE = Path.cwd()

raw_path = BASE.parent / "YSJ-002" / "Welding Data Set_01.xlsx"
ae_path  = BASE.parent / "YSJ-003" / "AE_C1_C2_date_row_detail.csv"

raw = pd.read_excel(raw_path, sheet_name="Raw data")
ae  = pd.read_csv(ae_path)

raw = raw.reset_index(drop=True)
raw["Original_Row"] = raw.index

# 컬럼 통일
raw["Force"] = pd.to_numeric(
    raw["weld force(bar)"], errors="coerce"
)

raw["Current"] = pd.to_numeric(
    raw["weld current(kA)"], errors="coerce"
)

raw["Voltage"] = pd.to_numeric(
    raw["weld Voltage(v)"], errors="coerce"
)

raw["Weld_Time"] = pd.to_numeric(
    raw["weld time(ms)"], errors="coerce"
)

raw["Date"] = pd.to_datetime(
    raw["working time"], errors="coerce"
).dt.strftime("%m/%d")


# ============================================================
# 2. B 날짜
# ============================================================

B_dates = [
    "03/25",
    "03/27",
    "03/31",
    "04/03"
]


# ============================================================
# 3. AE 후보 표시
# ============================================================

ae_rows = set(ae["Original_Row"])

raw["AE_Candidate"] = raw["Original_Row"].isin(ae_rows)


# ============================================================
# 4. 우리가 앞에서 확인한 반복 AE 구간 정의
#
# Force 3~10.xxx 쪽 AE 후보
# = Force 7~8 154개 + Other 97개
# ============================================================

target = raw[
    (raw["Date"].isin(B_dates)) &
    (raw["AE_Candidate"]) &
    (raw["Force"] >= 3) &
    (raw["Force"] < 11)
].copy()


print("=" * 90)
print("B 날짜별 반복 AE 후보")
print("=" * 90)

print(
    target.groupby("Date")
    .size()
)


# ============================================================
# 5. 각 날짜의 첫 후보 ~ 마지막 후보 사이
#    원본 데이터를 전부 추출
# ============================================================

blocks = {}

print("\n" + "=" * 90)
print("날짜별 반복구간 전체 범위")
print("=" * 90)


for date in B_dates:

    t = target[
        target["Date"] == date
    ].sort_values("Original_Row")

    if len(t) == 0:
        print(date, ": 후보 없음")
        continue

    start = int(t["Original_Row"].min())
    end   = int(t["Original_Row"].max())

    block = raw[
        (raw["Date"] == date) &
        (raw["Original_Row"] >= start) &
        (raw["Original_Row"] <= end)
    ].copy()

    block = block.sort_values("Original_Row")

    # 날짜 내부 상대 위치
    block["Relative_Row"] = np.arange(len(block))

    blocks[date] = block

    print(
        f"{date} | "
        f"Start={start} | "
        f"End={end} | "
        f"전체구간={len(block)}행 | "
        f"AE후보={block['AE_Candidate'].sum()}행"
    )


# ============================================================
# 6. 블록 길이 비교
# ============================================================

print("\n" + "=" * 90)
print("블록 길이")
print("=" * 90)

for date in B_dates:

    if date in blocks:
        print(
            f"{date}: {len(blocks[date])}행"
        )


# ============================================================
# 7. 4개 센서값이 완전히 동일한지 비교
# ============================================================

features = [
    "Force",
    "Current",
    "Voltage",
    "Weld_Time"
]

base_date = B_dates[0]
base = blocks[base_date]


print("\n" + "=" * 90)
print("03/25 기준 4개 센서 전체 순서 동일 여부")
print("=" * 90)


for date in B_dates[1:]:

    compare = blocks[date]

    if len(base) != len(compare):

        print(
            f"{base_date} vs {date}"
            f" → 길이 다름 "
            f"({len(base)} vs {len(compare)})"
        )

        continue

    print(f"\n{base_date} vs {date}")

    for feature in features:

        a = base[feature].to_numpy()
        b = compare[feature].to_numpy()

        same = np.array_equal(a, b)

        print(
            f"{feature:10s}: {same}"
        )


# ============================================================
# 8. 값이 완전히 같지 않더라도
#    행별 차이가 얼마나 작은지 계산
# ============================================================

print("\n" + "=" * 90)
print("행별 평균 절대 차이")
print("=" * 90)


for date in B_dates[1:]:

    compare = blocks[date]

    print(f"\n{base_date} vs {date}")

    if len(base) != len(compare):

        print("길이가 달라 직접 비교 제외")
        continue

    for feature in features:

        diff = np.abs(
            base[feature].to_numpy()
            -
            compare[feature].to_numpy()
        )

        print(
            f"{feature:10s}"
            f" | 평균차이={np.nanmean(diff):.6f}"
            f" | 최대차이={np.nanmax(diff):.6f}"
        )


# ============================================================
# 9. 행 단위로 4개 센서가 모두 동일한 개수
# ============================================================

print("\n" + "=" * 90)
print("4개 센서 완전 동일 행")
print("=" * 90)


for date in B_dates[1:]:

    compare = blocks[date]

    if len(base) != len(compare):
        continue

    same_row = (
        (base["Force"].to_numpy()
         == compare["Force"].to_numpy())
        &
        (base["Current"].to_numpy()
         == compare["Current"].to_numpy())
        &
        (base["Voltage"].to_numpy()
         == compare["Voltage"].to_numpy())
        &
        (base["Weld_Time"].to_numpy()
         == compare["Weld_Time"].to_numpy())
    )

    same_count = same_row.sum()

    print(
        f"{base_date} vs {date}"
        f" : {same_count}/{len(base)}"
        f" ({same_count/len(base)*100:.2f}%)"
    )


# ============================================================
# 10. 각 날짜 블록의 센서 평균
# ============================================================

print("\n" + "=" * 90)
print("날짜별 반복구간 평균")
print("=" * 90)

summary = []

for date in B_dates:

    block = blocks[date]

    summary.append({
        "Date": date,
        "Rows": len(block),
        "Force": block["Force"].mean(),
        "Current": block["Current"].mean(),
        "Voltage": block["Voltage"].mean(),
        "Weld_Time": block["Weld_Time"].mean(),
        "AE_Count": int(block["AE_Candidate"].sum())
    })

summary = pd.DataFrame(summary)

print(
    summary.round(4).to_string(index=False)
)


# ============================================================
# 11. 처음 30행 실제 순서 확인
# ============================================================

print("\n" + "=" * 90)
print("03/25 반복구간 처음 30행")
print("=" * 90)

print(
    blocks["03/25"][
        [
            "Relative_Row",
            "Original_Row",
            "Force",
            "Current",
            "Voltage",
            "Weld_Time",
            "AE_Candidate"
        ]
    ]
    .head(30)
    .to_string(index=False)
)


# ============================================================
# 12. 최종 판단용 출력
# ============================================================

print("\n" + "=" * 90)
print("확인할 것")
print("=" * 90)

print("""
1. 4개 날짜의 전체 블록 길이가 같은가?

2. Force 순서가 동일한가?

3. Current / Voltage / Weld Time도 동일한가?

4. 4개 센서가 완전히 동일한 행이 몇 %인가?

결과가 거의 동일하면:

→ B공정에는 날짜가 달라도 반복되는
  하나의 고정된 생산 시퀀스가 존재할 가능성이 큼.

그리고 AE는 그 시퀀스 중 일반적인 Force 2.xxx와
다른 부분을 높은 복원오차 후보로 잡았다고 해석할 수 있음.
""")

B 날짜별 반복 AE 후보
Date
03/25    251
03/27    251
03/31    251
04/03    251
dtype: int64

날짜별 반복구간 전체 범위
03/25 | Start=1877 | End=2174 | 전체구간=298행 | AE후보=254행
03/27 | Start=4691 | End=4988 | 전체구간=298행 | AE후보=254행
03/31 | Start=7801 | End=8098 | 전체구간=298행 | AE후보=254행
04/03 | Start=10877 | End=11174 | 전체구간=298행 | AE후보=254행

블록 길이
03/25: 298행
03/27: 298행
03/31: 298행
04/03: 298행

03/25 기준 4개 센서 전체 순서 동일 여부

03/25 vs 03/27
Force     : True
Current   : True
Voltage   : True
Weld_Time : True

03/25 vs 03/31
Force     : True
Current   : True
Voltage   : True
Weld_Time : True

03/25 vs 04/03
Force     : True
Current   : True
Voltage   : True
Weld_Time : True

행별 평균 절대 차이

03/25 vs 03/27
Force      | 평균차이=0.000000 | 최대차이=0.000000
Current    | 평균차이=0.000000 | 최대차이=0.000000
Voltage    | 평균차이=0.000000 | 최대차이=0.000000
Weld_Time  | 평균차이=0.000000 | 최대차이=0.000000

03/25 vs 03/31
Force      | 평균차이=0.000000 | 최대차이=0.000000
Current    | 평균차이=0.000000 | 최대차이=0.000000
Voltage    | 평균차이=0.000000 | 최대차이=0.000000


In [17]:
# ============================================================
# B 298행 반복블록
# AE 후보 254행 vs AE 비후보 44행 비교
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. 03/25의 298행 블록 사용
#    4개 B 날짜가 100% 동일하므로 한 블록만 분석
# ------------------------------------------------------------

block = blocks["03/25"].copy()

print("=" * 80)
print("B 298행 반복블록 - AE 후보 / 비후보 비교")
print("=" * 80)

print("전체 :", len(block))
print("AE 후보 :", int(block["AE_Candidate"].sum()))
print("AE 비후보 :", int((~block["AE_Candidate"]).sum()))


# ------------------------------------------------------------
# 2. 후보 / 비후보 센서 평균
# ------------------------------------------------------------

features = ["Force", "Current", "Voltage", "Weld_Time"]

print("\n" + "=" * 80)
print("AE 후보 vs AE 비후보 평균")
print("=" * 80)

for col in features:

    candidate = block.loc[
        block["AE_Candidate"], col
    ]

    normal = block.loc[
        ~block["AE_Candidate"], col
    ]

    print(f"\n[{col}]")

    print(
        f"AE 후보   | "
        f"평균={candidate.mean():.4f} | "
        f"최소={candidate.min():.4f} | "
        f"최대={candidate.max():.4f}"
    )

    print(
        f"AE 비후보 | "
        f"평균={normal.mean():.4f} | "
        f"최소={normal.min():.4f} | "
        f"최대={normal.max():.4f}"
    )


# ------------------------------------------------------------
# 3. 298행 전체에서 후보/비후보가 어디에 위치하는지
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("298행 순서")
print("=" * 80)

display_df = block[
    [
        "Relative_Row",
        "Force",
        "Current",
        "Voltage",
        "Weld_Time",
        "AE_Candidate"
    ]
].copy()

display_df["AE"] = np.where(
    display_df["AE_Candidate"],
    "후보",
    "비후보"
)

print(
    display_df[
        [
            "Relative_Row",
            "Force",
            "Current",
            "Voltage",
            "Weld_Time",
            "AE"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 4. 비후보 44행만 따로 출력
# ------------------------------------------------------------

non_ae = block[
    ~block["AE_Candidate"]
][
    [
        "Relative_Row",
        "Original_Row",
        "Force",
        "Current",
        "Voltage",
        "Weld_Time"
    ]
].copy()

print("\n" + "=" * 80)
print("AE 비후보 44행")
print("=" * 80)

print(non_ae.to_string(index=False))


# ------------------------------------------------------------
# 5. 비후보가 연속해서 나타나는지 확인
# ------------------------------------------------------------

non_ae = non_ae.sort_values("Relative_Row")

non_ae["Gap"] = non_ae["Relative_Row"].diff()

non_ae["New_Block"] = (
    non_ae["Gap"].isna() |
    (non_ae["Gap"] != 1)
)

non_ae["Block"] = non_ae["New_Block"].cumsum()

non_ae_blocks = (
    non_ae
    .groupby("Block")
    .agg(
        Start=("Relative_Row", "min"),
        End=("Relative_Row", "max"),
        Length=("Relative_Row", "size"),
        Force_Min=("Force", "min"),
        Force_Max=("Force", "max"),
        Current_Mean=("Current", "mean"),
        Voltage_Mean=("Voltage", "mean"),
        WeldTime_Mean=("Weld_Time", "mean")
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("AE 비후보 연속구간")
print("=" * 80)

print(
    non_ae_blocks
    .round(4)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Force 구간별 AE 후보율
# ------------------------------------------------------------

def force_group(x):

    if 2 <= x < 3:
        return "2.xxx"
    elif 3 <= x < 4:
        return "3.xxx"
    elif 4 <= x < 5:
        return "4.xxx"
    elif 5 <= x < 6:
        return "5.xxx"
    elif 6 <= x < 7:
        return "6.xxx"
    elif 7 <= x < 8:
        return "7.xxx"
    elif 8 <= x < 9:
        return "8.xxx"
    elif 9 <= x < 10:
        return "9.xxx"
    elif 10 <= x < 11:
        return "10.xxx"
    else:
        return "Other"


block["Force_Group"] = block["Force"].apply(force_group)

force_result = (
    block.groupby("Force_Group")
    .agg(
        전체행=("AE_Candidate", "size"),
        AE후보=("AE_Candidate", "sum")
    )
)

force_result["AE후보율(%)"] = (
    force_result["AE후보"]
    / force_result["전체행"]
    * 100
).round(2)

print("\n" + "=" * 80)
print("Force 구간별 AE 후보율")
print("=" * 80)

print(force_result.to_string())


# ------------------------------------------------------------
# 7. 핵심 확인
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("확인할 것")
print("=" * 80)

print("""
1. AE 비후보 44행이 298행 중 어디에 몰려 있는가?

2. 비후보가 특정 Force 구간에 몰려 있는가?

3. 후보와 비후보에서
   Force / Current / Voltage / Weld Time 중
   어떤 값의 차이가 가장 큰가?

→ 이것으로 AE가 298행 반복블록 중
  왜 254행만 높은 복원오차로 잡았는지 확인.
""")

B 298행 반복블록 - AE 후보 / 비후보 비교
전체 : 298
AE 후보 : 254
AE 비후보 : 44

AE 후보 vs AE 비후보 평균

[Force]
AE 후보   | 평균=7.2492 | 최소=2.7900 | 최대=10.5400
AE 비후보 | 평균=4.5730 | 최소=2.8600 | 최대=6.9300

[Current]
AE 후보   | 평균=14.8491 | 최소=14.7100 | 최대=15.0700
AE 비후보 | 평균=14.8180 | 최소=14.7300 | 최대=14.9900

[Voltage]
AE 후보   | 평균=2.7218 | 최소=2.4640 | 최대=2.8610
AE 비후보 | 평균=2.7451 | 최소=2.6930 | 최대=2.7760

[Weld_Time]
AE 후보   | 평균=71.7087 | 최소=70.0000 | 최대=73.0000
AE 비후보 | 평균=71.7273 | 최소=71.0000 | 최대=73.0000

298행 순서
 Relative_Row  Force  Current  Voltage  Weld_Time  AE
            0  10.54    14.75    2.476       72.0  후보
            1  10.54    14.76    2.476       72.0  후보
            2   9.27    14.76    2.474       72.0  후보
            3   9.27    14.78    2.474       72.0  후보
            4   9.07    14.78    2.468       72.0  후보
            5   9.07    14.75    2.468       72.0  후보
            6   9.01    14.75    2.464       71.0  후보
            7   9.01    14.83    2.464       71.0  후보
            8   9.

In [18]:
# ============================================================
# 298행 반복블록에서 AE가 어떤 변수를 크게 틀렸는지 확인
# Current / Voltage / Weld_Time 복원오차 비교
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. AE 결과 파일 불러오기
# ------------------------------------------------------------

ae_path = Path(r"C:\Kamp-2026\experiments\YSJ\YSJ-003\AE_C1_C2_date_row_detail.csv")

ae = pd.read_csv(ae_path)

# 날짜 형식 통일
ae["Date"] = pd.to_datetime(ae["Date"]).dt.strftime("%m/%d")

# ------------------------------------------------------------
# 2. 03/25의 298행 반복블록
# ------------------------------------------------------------

block = blocks["03/25"].copy()

start_row = block["Original_Row"].min()
end_row   = block["Original_Row"].max()

print("=" * 80)
print("298행 반복블록 AE 변수별 복원오차 분석")
print("=" * 80)

print("블록 시작:", start_row)
print("블록 끝  :", end_row)
print("전체 행  :", len(block))


# ------------------------------------------------------------
# 3. AE 결과와 298행 블록 연결
# ------------------------------------------------------------

block_ae = ae[
    (ae["Date"] == "03/25") &
    (ae["Original_Row"] >= start_row) &
    (ae["Original_Row"] <= end_row)
].copy()

print("\nAE 후보:", len(block_ae))


# ------------------------------------------------------------
# 4. 변수별 평균 복원오차
# ------------------------------------------------------------

error_cols = [
    "Current_Error",
    "Voltage_Error",
    "Weld_Time_Error"
]

print("\n" + "=" * 80)
print("변수별 평균 복원오차")
print("=" * 80)

for col in error_cols:

    print(
        f"{col:20s} | "
        f"평균={block_ae[col].mean():.4f} | "
        f"중앙값={block_ae[col].median():.4f} | "
        f"최대={block_ae[col].max():.4f}"
    )


# ------------------------------------------------------------
# 5. 각 행에서 가장 큰 복원오차 변수
# ------------------------------------------------------------

block_ae["Dominant_Error"] = (
    block_ae[error_cols]
    .idxmax(axis=1)
    .str.replace("_Error", "", regex=False)
)

dominant_count = block_ae["Dominant_Error"].value_counts()

dominant_rate = (
    block_ae["Dominant_Error"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n" + "=" * 80)
print("각 행에서 가장 큰 복원오차 변수")
print("=" * 80)

for feature in dominant_count.index:

    print(
        f"{feature:12s} : "
        f"{dominant_count[feature]}행 "
        f"({dominant_rate[feature]:.2f}%)"
    )


# ------------------------------------------------------------
# 6. Force 구간별로 어떤 오차가 큰지
# ------------------------------------------------------------

def force_group(x):

    if 2 <= x < 3:
        return "2.xxx"
    elif 3 <= x < 4:
        return "3.xxx"
    elif 4 <= x < 5:
        return "4.xxx"
    elif 5 <= x < 6:
        return "5.xxx"
    elif 6 <= x < 7:
        return "6.xxx"
    elif 7 <= x < 8:
        return "7.xxx"
    elif 8 <= x < 9:
        return "8.xxx"
    elif 9 <= x < 10:
        return "9.xxx"
    elif 10 <= x < 11:
        return "10.xxx"
    else:
        return "Other"


block_ae["Force_Group"] = block_ae["Force"].apply(force_group)

force_error = (
    block_ae
    .groupby("Force_Group")[error_cols]
    .mean()
    .round(4)
)

print("\n" + "=" * 80)
print("Force 구간별 평균 복원오차")
print("=" * 80)

print(force_error.to_string())


# ------------------------------------------------------------
# 7. Force 구간별 dominant error
# ------------------------------------------------------------

force_dominant = pd.crosstab(
    block_ae["Force_Group"],
    block_ae["Dominant_Error"]
)

print("\n" + "=" * 80)
print("Force 구간 × 가장 큰 복원오차 변수")
print("=" * 80)

print(force_dominant.to_string())


# ------------------------------------------------------------
# 8. 전체 Reconstruction Error와 Force 관계
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("Force와 Reconstruction Error 관계")
print("=" * 80)

pearson = block_ae[
    ["Force", "Reconstruction_Error"]
].corr(method="pearson").iloc[0, 1]

spearman = block_ae[
    ["Force", "Reconstruction_Error"]
].corr(method="spearman").iloc[0, 1]

print("Pearson :", round(pearson, 4))
print("Spearman:", round(spearman, 4))


# ------------------------------------------------------------
# 9. 핵심 요약
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("확인할 것")
print("=" * 80)

print("""
1. Current / Voltage / Weld_Time 중
   어떤 복원오차가 가장 큰가?

2. Dominant_Error에서
   특정 변수가 대부분을 차지하는가?

3. Force 7~10 구간에서
   어떤 변수의 복원오차가 커지는가?

4. Force와 전체 Reconstruction Error의
   Pearson / Spearman 관계가 강한가?

주의:
현재 CSV에는 Force_Error가 없으므로
'Force 자체의 복원오차'는 이 코드에서 비교하지 않음.

→ 이번 분석은
  높은 Force 구간에서 AE가 Current / Voltage / Weld_Time 중
  무엇을 비정상적으로 복원하고 있는지 확인하는 단계.
""")

298행 반복블록 AE 변수별 복원오차 분석
블록 시작: 1877
블록 끝  : 2174
전체 행  : 298

AE 후보: 254

변수별 평균 복원오차
Current_Error        | 평균=2.8185 | 중앙값=1.4572 | 최대=13.1441
Voltage_Error        | 평균=9.4966 | 중앙값=4.1395 | 최대=91.7467
Weld_Time_Error      | 평균=1.0434 | 중앙값=0.1904 | 최대=7.3909

각 행에서 가장 큰 복원오차 변수
Voltage      : 165행 (64.96%)
Current      : 76행 (29.92%)
Weld_Time    : 13행 (5.12%)

Force 구간별 평균 복원오차
             Current_Error  Voltage_Error  Weld_Time_Error
Force_Group                                               
10.xxx              0.2063        82.8020           0.1904
2.xxx               5.9198         4.0843           1.8494
3.xxx               2.1995         8.9141           1.9520
4.xxx               0.7279        20.8200           0.6052
5.xxx               2.9363        19.8494           1.3684
6.xxx               3.0973         3.9265           1.1179
7.xxx               3.3172         3.7654           0.9813
8.xxx               1.6844         7.7416           1.0813
9.xxx               0.57

In [19]:
import pandas as pd
import numpy as np

# ============================================================
# 1. 날짜별 공정 구분
# ============================================================

process_data = pd.DataFrame({
    "Date": [
        "03/24", "03/25", "03/26",
        "03/27", "03/30", "03/31",
        "04/02", "04/03", "04/07"
    ],

    "Process": [
        "A", "B", "A",
        "B", "A", "B",
        "A", "B", "A"
    ]
})

# B 날짜에는 동일한 298행 반복블록 존재
process_data["Repeat_Block"] = np.where(
    process_data["Process"] == "B",
    1,
    0
)


# ============================================================
# 2. 실제 불량 데이터
#
# 앞에서 확인한 값만 입력
# 아직 값이 없는 날짜는 NaN 처리
# ============================================================

defect_data = pd.DataFrame({

    "Date": [
        "03/24",
        "03/25",
        "03/31",
        "04/03"
    ],

    "Pitting": [
        2,
        3,
        2,
        2
    ],

    "Lack_Weld": [
        1,
        2,
        3,
        1
    ],

    "Crack": [
        1,
        3,
        np.nan,
        1
    ]
})


# ============================================================
# 3. 공정정보 + 실제불량 결합
# ============================================================

result = process_data.merge(
    defect_data,
    on="Date",
    how="left"
)

result["Total_Defect"] = (
    result[
        ["Pitting", "Lack_Weld", "Crack"]
    ]
    .sum(axis=1, min_count=1)
)


print("=" * 80)
print("날짜별 공정패턴 × 실제 불량")
print("=" * 80)

print(result.to_string(index=False))


# ============================================================
# 4. 실제 불량 정보가 존재하는 날짜만 사용
# ============================================================

valid = result[
    result[
        ["Pitting", "Lack_Weld", "Crack"]
    ].notna().any(axis=1)
].copy()


print("\n" + "=" * 80)
print("실제 불량 데이터가 있는 날짜")
print("=" * 80)

print(valid.to_string(index=False))


# ============================================================
# 5. A vs B 실제 불량 평균
# ============================================================

print("\n" + "=" * 80)
print("A/B 공정별 실제 불량 평균")
print("=" * 80)

summary = (
    valid
    .groupby("Process")[
        [
            "Pitting",
            "Lack_Weld",
            "Crack",
            "Total_Defect"
        ]
    ]
    .mean()
    .round(3)
)

print(summary.to_string())


# ============================================================
# 6. A/B 실제 불량 총합
# ============================================================

print("\n" + "=" * 80)
print("A/B 공정별 실제 불량 총합")
print("=" * 80)

total = (
    valid
    .groupby("Process")[
        [
            "Pitting",
            "Lack_Weld",
            "Crack",
            "Total_Defect"
        ]
    ]
    .sum()
)

print(total.to_string())


# ============================================================
# 7. 날짜별 직접 비교
# ============================================================

print("\n" + "=" * 80)
print("날짜별 상세")
print("=" * 80)

for _, row in valid.iterrows():

    print(
        f"{row['Date']} | "
        f"공정={row['Process']} | "
        f"298행블록={'있음' if row['Repeat_Block'] == 1 else '없음'} | "
        f"파임={row['Pitting']} | "
        f"용접부족={row['Lack_Weld']} | "
        f"크랙={row['Crack']} | "
        f"총불량={row['Total_Defect']}"
    )


# ============================================================
# 8. 298행 반복블록과 불량의 단순 상관
# ============================================================

print("\n" + "=" * 80)
print("298행 반복블록 존재 여부 ↔ 실제 불량 상관")
print("=" * 80)

for col in [
    "Pitting",
    "Lack_Weld",
    "Crack",
    "Total_Defect"
]:

    temp = valid[
        ["Repeat_Block", col]
    ].dropna()

    if len(temp) >= 2 and temp["Repeat_Block"].nunique() >= 2:

        spearman = temp.corr(
            method="spearman"
        ).iloc[0, 1]

        print(
            f"{col:15s} : "
            f"Spearman = {spearman:.4f}"
        )

    else:

        print(
            f"{col:15s} : 계산 불가"
        )


# ============================================================
# 9. 현재 데이터 한계
# ============================================================

print("\n" + "=" * 80)
print("해석할 때 확인할 것")
print("=" * 80)

print("""
1. A와 B에서 실제 불량 개수가 차이나는가?

2. B(298행 반복블록 존재)에서
   특정 불량이 반복적으로 증가하는가?

3. 03/25만 불량이 높은 것인지,
   B 날짜 전체에서 공통적으로 높은 것인지 확인.

주의:
현재 입력된 실제 불량 데이터는
03/24, 03/25, 03/31, 04/03뿐임.

따라서 표본이 매우 적으므로
상관계수만으로 298행 블록이 불량의 원인이라고
판단하면 안 됨.

목표:
'298행 반복블록 = 불량'을 가정하는 것이 아니라,
B 반복공정과 실제 불량 사이에
추가 분석할 만한 관계가 있는지 확인.
""")

날짜별 공정패턴 × 실제 불량
 Date Process  Repeat_Block  Pitting  Lack_Weld  Crack  Total_Defect
03/24       A             0      2.0        1.0    1.0           4.0
03/25       B             1      3.0        2.0    3.0           8.0
03/26       A             0      NaN        NaN    NaN           NaN
03/27       B             1      NaN        NaN    NaN           NaN
03/30       A             0      NaN        NaN    NaN           NaN
03/31       B             1      2.0        3.0    NaN           5.0
04/02       A             0      NaN        NaN    NaN           NaN
04/03       B             1      2.0        1.0    1.0           4.0
04/07       A             0      NaN        NaN    NaN           NaN

실제 불량 데이터가 있는 날짜
 Date Process  Repeat_Block  Pitting  Lack_Weld  Crack  Total_Defect
03/24       A             0      2.0        1.0    1.0           4.0
03/25       B             1      3.0        2.0    3.0           8.0
03/31       B             1      2.0        3.0    NaN           5.0

In [21]:
# ============================================================
# 공정상태별 AutoEncoder
#
# 목적:
# 기존 AE가 B의 298행 반복 공정블록을 이상으로 과검출한 문제를
# 공정상태별 AE 학습으로 줄일 수 있는지 확인
#
# 비교:
# 기존 AE : 298행 중 254행 = 85.2% 이상 후보
# 개선 AE : 298행 중 몇 행이 이상으로 남는가?
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 0. 설정
# ============================================================

np.random.seed(42)
tf.random.set_seed(42)

FILE = Path(
    r"C:\Kamp-2026\experiments\YSJ\YSJ-002\Welding Data Set_01.xlsx"
)

SHEET = "Raw data"

FEATURES = [
    "weld force(bar)",
    "weld current(kA)",
    "weld Voltage(v)",
    "weld time(ms)"
]

DATE_COL = "working time"

# 기존 결과
OLD_BLOCK_AE = 254
BLOCK_LENGTH = 298
OLD_RATE = OLD_BLOCK_AE / BLOCK_LENGTH * 100


# ============================================================
# 1. 데이터 불러오기
# ============================================================

df = pd.read_excel(FILE, sheet_name=SHEET)

df["Original_Row"] = np.arange(len(df))

df["Date"] = (
    pd.to_datetime(df[DATE_COL])
    .dt.strftime("%m/%d")
)

df = df.dropna(subset=FEATURES).copy()


print("=" * 80)
print("원본 데이터")
print("=" * 80)

print("전체 행:", len(df))
print("날짜:", df["Date"].unique())


# ============================================================
# 2. 298행 반복블록 지정
#
# 앞에서 확인된 블록
# 03/25 : 1877 ~ 2174
# 03/27 : 4691 ~ 4988
# 03/31 : 7801 ~ 8098
# 04/03 : 10877 ~ 11174
# ============================================================

blocks = [
    (1877, 2174),
    (4691, 4988),
    (7801, 8098),
    (10877, 11174)
]

df["Process_State"] = "General"

for start, end in blocks:

    mask = (
        (df["Original_Row"] >= start) &
        (df["Original_Row"] <= end)
    )

    df.loc[mask, "Process_State"] = "B_Repeat_298"


print("\n" + "=" * 80)
print("공정상태별 행 개수")
print("=" * 80)

print(df["Process_State"].value_counts())


# ============================================================
# 3. AE 함수
# ============================================================

def run_autoencoder(data, state_name):

    print("\n")
    print("=" * 80)
    print(state_name, "AutoEncoder")
    print("=" * 80)

    X = data[FEATURES].values

    # --------------------------------------------------------
    # Train / Test
    # --------------------------------------------------------

    X_train, X_test = train_test_split(
        X,
        test_size=0.20,
        random_state=42
    )

    # --------------------------------------------------------
    # Scaling
    # scaler는 train에만 fit
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_all_scaled = scaler.transform(X)

    # --------------------------------------------------------
    # AutoEncoder
    # --------------------------------------------------------

    model = Sequential([
        Dense(
            8,
            activation="relu",
            input_shape=(len(FEATURES),)
        ),

        Dense(
            4,
            activation="relu"
        ),

        Dense(
            2,
            activation="relu"
        ),

        Dense(
            4,
            activation="relu"
        ),

        Dense(
            8,
            activation="relu"
        ),

        Dense(
            len(FEATURES),
            activation="linear"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )

    model.fit(
        X_train_scaled,
        X_train_scaled,

        validation_split=0.20,

        epochs=200,
        batch_size=32,

        callbacks=[early_stop],

        verbose=0
    )

    # --------------------------------------------------------
    # Train reconstruction error
    # --------------------------------------------------------

    train_pred = model.predict(
        X_train_scaled,
        verbose=0
    )

    train_error = np.mean(
        np.square(
            X_train_scaled - train_pred
        ),
        axis=1
    )

    # --------------------------------------------------------
    # Threshold
    #
    # 정상 학습 데이터 reconstruction error의
    # 95 percentile 사용
    # --------------------------------------------------------

    threshold = np.percentile(
        train_error,
        95
    )

    # --------------------------------------------------------
    # 전체 데이터 reconstruction error
    # --------------------------------------------------------

    pred = model.predict(
        X_all_scaled,
        verbose=0
    )

    error = np.mean(
        np.square(
            X_all_scaled - pred
        ),
        axis=1
    )

    result = data.copy()

    result["AE_Error"] = error

    result["AE_Anomaly"] = (
        result["AE_Error"] > threshold
    )

    print("데이터:", len(result))
    print("Threshold:", round(threshold, 6))

    print(
        "이상 후보:",
        result["AE_Anomaly"].sum()
    )

    print(
        "이상 후보율:",
        round(
            result["AE_Anomaly"].mean() * 100,
            2
        ),
        "%"
    )

    return result, model, scaler, threshold


# ============================================================
# 4. 일반 공정 AE
# ============================================================

general = df[
    df["Process_State"] == "General"
].copy()

general_result, general_model, general_scaler, general_threshold = (
    run_autoencoder(
        general,
        "General"
    )
)


# ============================================================
# 5. B 298행 반복공정 전용 AE
# ============================================================

repeat = df[
    df["Process_State"] == "B_Repeat_298"
].copy()

repeat_result, repeat_model, repeat_scaler, repeat_threshold = (
    run_autoencoder(
        repeat,
        "B Repeat 298"
    )
)


# ============================================================
# 6. 298행 블록 날짜별 결과
# ============================================================

repeat_daily = (
    repeat_result
    .groupby("Date")
    .agg(
        Rows=("AE_Anomaly", "size"),
        AE_Anomaly=("AE_Anomaly", "sum")
    )
)

repeat_daily["AE_Rate_%"] = (
    repeat_daily["AE_Anomaly"]
    / repeat_daily["Rows"]
    * 100
).round(2)


print("\n")
print("=" * 80)
print("B 298행 반복블록 날짜별 개선 AE 결과")
print("=" * 80)

print(repeat_daily)


# ============================================================
# 7. 전체 반복블록 결과
# ============================================================

new_count = int(
    repeat_result["AE_Anomaly"].sum()
)

new_rate = (
    repeat_result["AE_Anomaly"].mean()
    * 100
)


print("\n")
print("=" * 80)
print("기존 AE vs 공정상태별 AE")
print("=" * 80)

print()

print("[기존 AE]")

print(
    f"298행 중 {OLD_BLOCK_AE}행"
)

print(
    f"이상 후보율 = {OLD_RATE:.2f}%"
)


print()

print("[개선 AE]")

print(
    f"반복블록 전체 {len(repeat_result)}행 중 "
    f"{new_count}행"
)

print(
    f"이상 후보율 = {new_rate:.2f}%"
)


print()

print(
    "과검출 감소:",
    round(
        OLD_RATE - new_rate,
        2
    ),
    "%p"
)


# ============================================================
# 8. 날짜별 패턴이 동일하므로
# 각 상대 위치가 반복해서 이상으로 잡히는지 확인
# ============================================================

repeat_result = repeat_result.sort_values(
    ["Date", "Original_Row"]
).copy()

repeat_result["Relative_Row"] = (
    repeat_result
    .groupby("Date")
    .cumcount()
)


position_summary = (
    repeat_result
    .groupby("Relative_Row")
    .agg(
        Dates=("Date", "nunique"),
        Anomaly_Count=("AE_Anomaly", "sum"),
        Mean_Error=("AE_Error", "mean")
    )
)


print("\n")
print("=" * 80)
print("개선 후에도 4개 날짜에서 반복적으로 이상인 위치")
print("=" * 80)

repeated_anomaly = position_summary[
    position_summary["Anomaly_Count"] == 4
]

print(
    "4/4 날짜에서 모두 이상:",
    len(repeated_anomaly),
    "개 위치"
)

if len(repeated_anomaly) > 0:

    print()

    print(
        repeated_anomaly
        .sort_values(
            "Mean_Error",
            ascending=False
        )
        .head(30)
    )


# ============================================================
# 9. 최종 요약
# ============================================================

print("\n")
print("=" * 80)
print("최종 판단")
print("=" * 80)

print(f"""
기존 AE
→ 298행 중 254행 이상
→ 85.23%

공정상태별 개선 AE
→ 반복공정 전체 {len(repeat_result)}행 중 {new_count}행 이상
→ {new_rate:.2f}%

확인할 것:

1. 기존 85.23%보다 크게 감소했는가?

2. 감소했다면
   → 기존 AE가 B 반복공정 자체를
     이상으로 과검출하고 있었다는 근거

3. 개선 후에도 반복적으로 이상으로 남는 위치가 있는가?

4. 특히 4개 B 날짜에서
   같은 Relative_Row가 계속 이상이면
   → 공정상태 자체가 아니라
     반복공정 내부의 특정 센서 상태를
     추가 분석할 가치가 있음
""")

원본 데이터
전체 행: 11939
날짜: <StringArray>
['03/24', '03/25', '03/26', '03/27', '03/30', '03/31', '04/02', '04/03',
 '04/07']
Length: 9, dtype: str

공정상태별 행 개수
Process_State
General         10747
B_Repeat_298     1192
Name: count, dtype: int64


General AutoEncoder


c:\Users\sj789\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


데이터: 10747
Threshold: 0.435226
이상 후보: 533
이상 후보율: 4.96 %


B Repeat 298 AutoEncoder


c:\Users\sj789\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


데이터: 1192
Threshold: 2.6888
이상 후보: 60
이상 후보율: 5.03 %


B 298행 반복블록 날짜별 개선 AE 결과
       Rows  AE_Anomaly  AE_Rate_%
Date                              
03/25   298          15       5.03
03/27   298          15       5.03
03/31   298          15       5.03
04/03   298          15       5.03


기존 AE vs 공정상태별 AE

[기존 AE]
298행 중 254행
이상 후보율 = 85.23%

[개선 AE]
반복블록 전체 1192행 중 60행
이상 후보율 = 5.03%

과검출 감소: 80.2 %p


개선 후에도 4개 날짜에서 반복적으로 이상인 위치
4/4 날짜에서 모두 이상: 15 개 위치

              Dates  Anomaly_Count  Mean_Error
Relative_Row                                  
0                 4              4    4.682797
1                 4              4    4.629387
6                 4              4    4.475813
7                 4              4    4.220951
9                 4              4    4.205723
8                 4              4    4.156332
5                 4              4    4.134021
2                 4              4    4.005117
4                 4              4    3.992265
3                 4 

In [22]:
# ============================================================
# B 298행 반복공정 Leave-One-Date-Out 검증
#
# 목적:
# 4개 B 날짜 중 3개 날짜만 학습하고
# 학습에 사용하지 않은 1개 날짜(298행)를 테스트
#
# → 랜덤 split 대신 날짜 단위로 완전히 분리
# → 새로운 날짜에서도 B 반복공정을 정상상태로 인식하는지 확인
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. 설정
# ============================================================

np.random.seed(42)
tf.random.set_seed(42)

FEATURES = [
    "weld force(bar)",
    "weld current(kA)",
    "weld Voltage(v)",
    "weld time(ms)"
]

B_DATES = [
    "03/25",
    "03/27",
    "03/31",
    "04/03"
]


# ============================================================
# 2. 앞 셀에서 만든 repeat 데이터 사용
# ============================================================

print("=" * 90)
print("B 298행 반복공정 데이터 확인")
print("=" * 90)

print("전체 행:", len(repeat))

print("\n날짜별 행 개수:")
print(repeat["Date"].value_counts().sort_index())


# ============================================================
# 3. 날짜별 Leave-One-Date-Out
# ============================================================

all_results = []
summary_results = []

for test_date in B_DATES:

    print("\n")
    print("=" * 90)
    print(f"TEST DATE = {test_date}")
    print("=" * 90)

    # --------------------------------------------------------
    # Train / Test 날짜 분리
    # --------------------------------------------------------

    train_df = repeat[
        repeat["Date"] != test_date
    ].copy()

    test_df = repeat[
        repeat["Date"] == test_date
    ].copy()

    train_dates = sorted(
        train_df["Date"].unique()
    )

    print("Train 날짜:", train_dates)
    print("Test 날짜 :", test_date)

    print(
        f"Train = {len(train_df)}행 | "
        f"Test = {len(test_df)}행"
    )


    # --------------------------------------------------------
    # 입력 데이터
    # --------------------------------------------------------

    X_train = train_df[FEATURES].values
    X_test = test_df[FEATURES].values


    # --------------------------------------------------------
    # Scaling
    # scaler는 Train에만 fit
    # --------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # --------------------------------------------------------
    # AutoEncoder
    # --------------------------------------------------------

    model = Sequential([

        Input(shape=(len(FEATURES),)),

        Dense(
            8,
            activation="relu"
        ),

        Dense(
            4,
            activation="relu"
        ),

        Dense(
            2,
            activation="relu"
        ),

        Dense(
            4,
            activation="relu"
        ),

        Dense(
            8,
            activation="relu"
        ),

        Dense(
            len(FEATURES),
            activation="linear"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )


    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )


    model.fit(
        X_train_scaled,
        X_train_scaled,

        validation_split=0.20,

        epochs=200,
        batch_size=32,

        callbacks=[early_stop],

        verbose=0
    )


    # --------------------------------------------------------
    # Train Reconstruction Error
    # --------------------------------------------------------

    train_pred = model.predict(
        X_train_scaled,
        verbose=0
    )

    train_error = np.mean(
        np.square(
            X_train_scaled - train_pred
        ),
        axis=1
    )


    # --------------------------------------------------------
    # Threshold = Train Error 95 percentile
    # --------------------------------------------------------

    threshold = np.percentile(
        train_error,
        95
    )


    # --------------------------------------------------------
    # Test Reconstruction Error
    # --------------------------------------------------------

    test_pred = model.predict(
        X_test_scaled,
        verbose=0
    )

    test_error = np.mean(
        np.square(
            X_test_scaled - test_pred
        ),
        axis=1
    )


    anomaly = (
        test_error > threshold
    )


    # --------------------------------------------------------
    # 결과 저장
    # --------------------------------------------------------

    temp = test_df.copy()

    temp["AE_Error_LODO"] = test_error
    temp["Threshold"] = threshold
    temp["AE_Anomaly_LODO"] = anomaly

    temp = temp.sort_values(
        "Original_Row"
    ).copy()

    temp["Relative_Row"] = np.arange(
        len(temp)
    )

    all_results.append(temp)


    anomaly_count = int(
        anomaly.sum()
    )

    anomaly_rate = (
        anomaly.mean() * 100
    )


    summary_results.append({

        "Test_Date": test_date,

        "Train_Rows": len(train_df),

        "Test_Rows": len(test_df),

        "Threshold": threshold,

        "AE_Anomaly": anomaly_count,

        "AE_Rate_%": anomaly_rate,

        "Mean_Test_Error": test_error.mean(),

        "Max_Test_Error": test_error.max()
    })


    print(
        "Threshold:",
        round(threshold, 6)
    )

    print(
        f"Test 이상 후보: "
        f"{anomaly_count}/{len(test_df)}"
    )

    print(
        f"Test 이상 후보율: "
        f"{anomaly_rate:.2f}%"
    )


# ============================================================
# 4. 4회 결과 합치기
# ============================================================

LODO_detail = pd.concat(
    all_results,
    ignore_index=True
)

LODO_summary = pd.DataFrame(
    summary_results
)


print("\n")
print("=" * 90)
print("Leave-One-Date-Out 최종 결과")
print("=" * 90)

print(
    LODO_summary
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 5. 전체 테스트 결과
# ============================================================

total_anomaly = int(
    LODO_detail["AE_Anomaly_LODO"].sum()
)

total_rows = len(
    LODO_detail
)

total_rate = (
    total_anomaly
    / total_rows
    * 100
)


print("\n")
print("=" * 90)
print("전체 날짜 통합 결과")
print("=" * 90)

print(
    f"전체 테스트 행: {total_rows}"
)

print(
    f"이상 후보: {total_anomaly}"
)

print(
    f"이상 후보율: {total_rate:.2f}%"
)


# ============================================================
# 6. 기존 AE와 비교
# ============================================================

OLD_RATE = 254 / 298 * 100


print("\n")
print("=" * 90)
print("기존 AE vs 날짜분리 개선 AE")
print("=" * 90)

print(
    f"기존 AE 반복블록 이상률 : "
    f"{OLD_RATE:.2f}%"
)

print(
    f"개선 AE 반복블록 이상률 : "
    f"{total_rate:.2f}%"
)

print(
    f"차이 : "
    f"{OLD_RATE - total_rate:.2f}%p 감소"
)


# ============================================================
# 7. 같은 상대위치가 4일 모두 이상인지 확인
# ============================================================

position_check = (
    LODO_detail
    .groupby("Relative_Row")
    .agg(
        Tested_Dates=("Date", "nunique"),
        Anomaly_Count=("AE_Anomaly_LODO", "sum"),
        Mean_Error=("AE_Error_LODO", "mean")
    )
)


same_position = position_check[
    position_check["Anomaly_Count"] == 4
]


print("\n")
print("=" * 90)
print("4번의 독립 테스트에서 모두 이상으로 남은 상대위치")
print("=" * 90)

print(
    "개수:",
    len(same_position)
)

if len(same_position) > 0:

    print(
        same_position
        .sort_values(
            "Mean_Error",
            ascending=False
        )
        .head(30)
        .round(4)
        .to_string()
    )


# ============================================================
# 8. 결과 저장
# ============================================================

LODO_summary.to_csv(
    "B_298_AE_LODO_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

LODO_detail.to_csv(
    "B_298_AE_LODO_detail.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n")
print("=" * 90)
print("확인할 것")
print("=" * 90)

print("""
1. 03/25, 03/27, 03/31, 04/03 각각
   학습에서 제외했는데도 이상률이 낮은가?

2. 4개 Test 날짜의 이상률이 비슷한가?

3. 기존 85.23%보다 크게 감소하는가?

4. 날짜를 완전히 분리했는데도 낮게 나온다면
   B의 298행 시퀀스를 별도 공정상태로 학습하는 방식이
   기존 AE의 반복공정 과검출을 줄인다는 근거가 됨.

5. 개선 후에도 계속 이상으로 남는 Relative_Row가 있다면
   그 위치의 Force / Current / Voltage / Weld Time을
   다음 단계에서 확인.
""")

B 298행 반복공정 데이터 확인
전체 행: 1192

날짜별 행 개수:
Date
03/25    298
03/27    298
03/31    298
04/03    298
Name: count, dtype: int64


TEST DATE = 03/25
Train 날짜: ['03/27', '03/31', '04/03']
Test 날짜 : 03/25
Train = 894행 | Test = 298행
Threshold: 0.504122
Test 이상 후보: 15/298
Test 이상 후보율: 5.03%


TEST DATE = 03/27
Train 날짜: ['03/25', '03/31', '04/03']
Test 날짜 : 03/27
Train = 894행 | Test = 298행
Threshold: 0.607714
Test 이상 후보: 15/298
Test 이상 후보율: 5.03%


TEST DATE = 03/31
Train 날짜: ['03/25', '03/27', '04/03']
Test 날짜 : 03/31
Train = 894행 | Test = 298행
Threshold: 1.230594
Test 이상 후보: 15/298
Test 이상 후보율: 5.03%


TEST DATE = 04/03
Train 날짜: ['03/25', '03/27', '03/31']
Test 날짜 : 04/03
Train = 894행 | Test = 298행
Threshold: 0.548053
Test 이상 후보: 15/298
Test 이상 후보율: 5.03%


Leave-One-Date-Out 최종 결과
Test_Date  Train_Rows  Test_Rows  Threshold  AE_Anomaly  AE_Rate_%  Mean_Test_Error  Max_Test_Error
    03/25         894        298     0.5041          15     5.0336           0.1316          1.0681
    03/27    